<a href="https://colab.research.google.com/github/nersot/project/blob/main/data_merge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount my google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Access files from my google drive
from pathlib import Path
BASE = Path("/content/drive/MyDrive/GRADTDA5621_Project_I") # THIS IS THE PATH TO THE FOLDER WITH THE FILES
RAW  = BASE / "Data_Sources" # THIS IS WHERE MY RAW FILES LIVE
RAW

Mounted at /content/drive


PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources')

In [ ]:
# List all files in my folder
list(RAW.glob("*"))

[PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/nanda_csv_readme.txt'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/nanda_grocery_Tract10_1990-2021_01P.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/nanda_grocery_ZCTA10_1990-2021_01P.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/nanda_grocery_Tract20_1990-2021_01P.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/nanda_grocery_ZCTA20_1990-2021_01P.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/Ruralurbancontinuumcodes2023.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/ACSST5Y2023.S1903-Column-Metadata.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/ACSST5Y2023.S1903-Table-Notes.txt'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/ACSST5Y2023.S1903-Data.csv')]

I am going to try a new strategy since the original ACS file was state level only and we need county level data. I downloaded the county level ACS file. I also selected ACS 2023 to align with RUCC 2023. The earliest data available for NaNDA is 2021. I will check how many counties are listed in each of the files (RUCC, NaNDA ZCTA20, NaNDA Tract20, and ACS COUNTY). I will use the file with the least listed counties to extract information from the remaining files.

In [ ]:
# Create output folder
BASE = Path("/content/drive/MyDrive/GRADTDA5621_Project_I")
RAW  = BASE / "Data_Sources"
OUT  = BASE / "Outputs"; OUT.mkdir(exist_ok=True)

In [ ]:
# Load ACS data
import pandas as pd

acs_path = RAW / "ACSST5Y2023.S1903-Data.csv"
acs = pd.read_csv(acs_path)

# Peek at first rows
print(acs.shape)
acs.head(3)

(3223, 243)


,GEO_ID,NAME,S1903_C01_001E,S1903_C01_001M,S1903_C01_002E,S1903_C01_002M,S1903_C01_003E,S1903_C01_003M,S1903_C01_004E,S1903_C01_004M,...,S1903_C03_036M,S1903_C03_037E,S1903_C03_037M,S1903_C03_038E,S1903_C03_038M,S1903_C03_039E,S1903_C03_039M,S1903_C03_040E,S1903_C03_040M,Unnamed: 242
0,Geography,Geographic Area Name,Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND...,Margin of Error!!Number!!HOUSEHOLD INCOME BY R...,Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND...,Margin of Error!!Number!!HOUSEHOLD INCOME BY R...,Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND...,Margin of Error!!Number!!HOUSEHOLD INCOME BY R...,Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND...,Margin of Error!!Number!!HOUSEHOLD INCOME BY R...,...,Margin of Error!!Median income (dollars)!!NONF...,Estimate!!Median income (dollars)!!NONFAMILY H...,Margin of Error!!Median income (dollars)!!NONF...,Estimate!!Median income (dollars)!!NONFAMILY H...,Margin of Error!!Median income (dollars)!!NONF...,Estimate!!Median income (dollars)!!NONFAMILY H...,Margin of Error!!Median income (dollars)!!NONF...,Estimate!!Median income (dollars)!!NONFAMILY H...,Margin of Error!!Median income (dollars)!!NONF...,NaN
1,0500000US01001,"Autauga County, Alabama",22523,431,16901,415,4198,308,13,22,...,2660,45509,22483,48909,7421,43900,5962,88750,13713,NaN
2,0500000US01003,"Baldwin County, Alabama",94642,1316,81281,1245,7647,544,285,110,...,2415,66490,25264,50819,5602,42586,3460,85794,15009,NaN


In [ ]:
# Load metadata
meta_path = RAW / "ACSST5Y2023.S1903-Column-Metadata.csv"
meta = pd.read_csv(meta_path)

print(meta.shape)
meta.head()

(242, 2)


,Column Name,Label
0,GEO_ID,Geography
1,NAME,Geographic Area Name
2,S1903_C01_001E,Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND...
3,S1903_C01_001M,Margin of Error!!Number!!HOUSEHOLD INCOME BY R...
4,S1903_C01_002E,Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND...


In [ ]:
# Drop row 0 and any unnamed columns
acs_data = acs.iloc[1:].reset_index(drop=True)
if "Unnamed: 242" in acs_data.columns:
    acs_data = acs_data.drop(columns=["Unnamed: 242"])

acs_data.head(5)

,GEO_ID,NAME,S1903_C01_001E,S1903_C01_001M,S1903_C01_002E,S1903_C01_002M,S1903_C01_003E,S1903_C01_003M,S1903_C01_004E,S1903_C01_004M,...,S1903_C03_036E,S1903_C03_036M,S1903_C03_037E,S1903_C03_037M,S1903_C03_038E,S1903_C03_038M,S1903_C03_039E,S1903_C03_039M,S1903_C03_040E,S1903_C03_040M
0,0500000US01001,"Autauga County, Alabama",22523,431,16901,415,4198,308,13,22,...,32570,2660,45509,22483,48909,7421,43900,5962,88750,13713
1,0500000US01003,"Baldwin County, Alabama",94642,1316,81281,1245,7647,544,285,110,...,32996,2415,66490,25264,50819,5602,42586,3460,85794,15009
2,0500000US01005,"Barbour County, Alabama",9080,315,4351,227,4091,293,20,20,...,22011,2219,49583,28479,22796,7049,22566,7203,-,**
3,0500000US01007,"Bibb County, Alabama",7571,288,5931,283,1460,248,83,54,...,19528,4208,93162,49213,28048,4027,26750,6800,-,**
4,0500000US01009,"Blount County, Alabama",21977,403,20210,494,161,66,71,41,...,19949,2731,46449,22440,38735,8409,29075,4171,81250,37693


In [ ]:
print(meta.columns.tolist())

['Column Name', 'Label']


In [ ]:
# Remove white space from headers and rename meta headers
meta.columns = meta.columns.str.strip()

if set(["Column Name","Label"]).issubset(meta.columns):
    meta = meta.rename(columns={"Column Name":"name","Label":"label"})

# Drop any empty rows
meta = meta.loc[:, ["name","label"]].dropna(how="all")
meta["name"]  = meta["name"].astype(str).str.strip()
meta["label"] = meta["label"].astype(str).str.strip()

print(meta.head(3))

             name                                              label
0          GEO_ID                                          Geography
1            NAME                               Geographic Area Name
2  S1903_C01_001E  Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND...


In [ ]:
# Creating dictionary using metadata for renaming headers in ACS table
label_map = dict(zip(meta["name"], meta["label"]))

# Add label_map to header and keep code
rename_map = {c: f"{c} | {label_map[c]}" for c in acs_data.columns if c in label_map}

# Apply rename
acs_wide_labeled = acs_data.rename(columns=rename_map)

acs_wide_labeled.head(3)

,GEO_ID | Geography,NAME | Geographic Area Name,S1903_C01_001E | Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households,S1903_C01_001M | Margin of Error!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households,S1903_C01_002E | Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!White,S1903_C01_002M | Margin of Error!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!White,S1903_C01_003E | Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Black or African American,S1903_C01_003M | Margin of Error!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Black or African American,S1903_C01_004E | Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!American Indian and Alaska Native,S1903_C01_004M | Margin of Error!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!American Indian and Alaska Native,...,S1903_C03_036E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder!!Living alone,S1903_C03_036M | Margin of Error!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder!!Living alone,S1903_C03_037E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder!!Not living alone,S1903_C03_037M | Margin of Error!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder!!Not living alone,S1903_C03_038E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder,S1903_C03_038M | Margin of Error!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder,S1903_C03_039E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder!!Living alone,S1903_C03_039M | Margin of Error!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder!!Living alone,S1903_C03_040E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder!!Not living alone,S1903_C03_040M | Margin of Error!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder!!Not living alone
0,0500000US01001,"Autauga County, Alabama",22523,431,16901,415,4198,308,13,22,...,32570,2660,45509,22483,48909,7421,43900,5962,88750,13713
1,0500000US01003,"Baldwin County, Alabama",94642,1316,81281,1245,7647,544,285,110,...,32996,2415,66490,25264,50819,5602,42586,3460,85794,15009
2,0500000US01005,"Barbour County, Alabama",9080,315,4351,227,4091,293,20,20,...,22011,2219,49583,28479,22796,7049,22566,7203,-,**


In [ ]:
# Select dataframe to work with
df = acs_wide_labeled.copy()

# Select only the code portion of the header (anything before |)
def code_of(col: str) -> str:
    return col.split(" | ", 1)[0]  # "S1903_C02_001E | ..." -> "S1903_C02_001E"

# Find ID columns and rename them to GEO_ID and NAME
id_cols = [c for c in df.columns if code_of(c) in {"GEO_ID", "NAME"}]
print("ID cols (detected):", id_cols)
rename_ids = {c: code_of(c) for c in id_cols}
df = df.rename(columns=rename_ids)

# Just confirming they were renamed
print([c for c in df.columns if c in {"GEO_ID","NAME"}])

ID cols (detected): ['GEO_ID | Geography', 'NAME | Geographic Area Name']
['GEO_ID', 'NAME']


In [ ]:
# Separating the ID columns from the data columns and creating a dictionary of the raw ACS code
all_cols = [c for c in df.columns if c not in {"GEO_ID","NAME"}]
codes    = {c: code_of(c) for c in all_cols}

# Keep only the estimates and removes the margins of error
estimate_cols = [c for c in all_cols if codes[c].endswith("E")]

# Eliminate raw count columns and only keep percentages and median incomes
keep_est_cols = [c for c in estimate_cols if ("_C02_" in codes[c] or "_C03_" in codes[c])]

# Just confirming again because I am paranoid :')
print("Columns kept (C02/C03 Estimates):", len(keep_est_cols))

Columns kept (C02/C03 Estimates): 80


In [ ]:
# Defining the trimmed data frame
acs_trim = df[["GEO_ID","NAME"] + keep_est_cols].copy()

# Convert all values to numeric
for c in keep_est_cols:
    acs_trim[c] = pd.to_numeric(acs_trim[c], errors="coerce")

# Mic check 1, 2
print("Trimmed shape:", acs_trim.shape)
acs_trim.head(3)

Trimmed shape: (3222, 82)


,GEO_ID,NAME,S1903_C02_001E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households,S1903_C02_002E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!White,S1903_C02_003E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Black or African American,S1903_C02_004E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!American Indian and Alaska Native,S1903_C02_005E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Asian,S1903_C02_006E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Native Hawaiian and Other Pacific Islander,S1903_C02_007E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Some other race,S1903_C02_008E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!Two or more races,...,S1903_C03_031E | Estimate!!Median income (dollars)!!FAMILY INCOME BY NUMBER OF EARNERS!!1 earner,S1903_C03_032E | Estimate!!Median income (dollars)!!FAMILY INCOME BY NUMBER OF EARNERS!!2 earners,S1903_C03_033E | Estimate!!Median income (dollars)!!FAMILY INCOME BY NUMBER OF EARNERS!!3 or more earners,S1903_C03_034E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households,S1903_C03_035E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder,S1903_C03_036E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder!!Living alone,S1903_C03_037E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder!!Not living alone,S1903_C03_038E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder,S1903_C03_039E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder!!Living alone,S1903_C03_040E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder!!Not living alone
0,0500000US01001,"Autauga County, Alabama",22523,75.0,18.6,0.1,1.0,0.0,1.0,4.2,...,61202.0,103776.0,151521.0,38506.0,33293.0,32570.0,45509.0,48909.0,43900.0,88750.0
1,0500000US01003,"Baldwin County, Alabama",94642,85.9,8.1,0.3,0.6,0.0,1.8,3.3,...,67157.0,118730.0,142571.0,41784.0,34592.0,32996.0,66490.0,50819.0,42586.0,85794.0
2,0500000US01005,"Barbour County, Alabama",9080,47.9,45.1,0.2,0.6,0.0,4.2,2.0,...,44600.0,88924.0,99850.0,23054.0,23114.0,22011.0,49583.0,22796.0,22566.0,NaN


Looks like C02_001E is giving us counts instead of a percentage. It should be a sum of all the percentages under columns C02. Will probably drop this column and verify that the C02 columns add up to 100 for their respective rows.

In [ ]:
# Pulling code from the code_of dictionary to check what C01_001E and C02_001E say
def code_of(col):
    return col.split(" | ", 1)[0]

# Finding the exact code
c01_total = next(c for c in df.columns if code_of(c) == "S1903_C01_001E")
c02_total = next(c for c in df.columns if code_of(c) == "S1903_C02_001E")

print("Labels:")
print(" C01 total:", c01_total)
print(" C02 total:", c02_total)

print("\nFirst 5 values (C01=counts, C02=percent total):")
print(df[[c01_total, c02_total]].head())

Labels:
 C01 total: S1903_C01_001E | Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households
 C02 total: S1903_C02_001E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households

First 5 values (C01=counts, C02=percent total):
  S1903_C01_001E | Estimate!!Number!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households  \
0                                              22523                                                                     
1                                              94642                                                                     
2                                               9080                                                                     
3                                               7571                                                                     
4                                              21977         

In [ ]:
# Confirming subgroups add up to ~100
import re, pandas as pd

# pick subgroup percent columns: C02_002E..C02_008E
sub_cols = [c for c in df.columns if re.match(r"^S1903_C02_00[2-8]E$", code_of(c))]
tmp = df[sub_cols].apply(pd.to_numeric, errors="coerce")
df["__pct_sum"] = tmp.sum(axis=1)
print(df[["NAME", "__pct_sum"]].head())  # should be ~100

                      NAME  __pct_sum
0  Autauga County, Alabama       99.9
1  Baldwin County, Alabama      100.0
2  Barbour County, Alabama      100.0
3     Bibb County, Alabama       99.9
4   Blount County, Alabama       99.9


In [ ]:
# Dropping C02_001E
if any(code_of(c) == "S1903_C02_001E" for c in acs_trim.columns):
    col_c02_total = next(c for c in acs_trim.columns if code_of(c) == "S1903_C02_001E")
    acs_trim = acs_trim.drop(columns=[col_c02_total])

# Adding in C01_001E
col_c01_total = next(c for c in df.columns if code_of(c) == "S1903_C01_001E")
acs_trim = acs_trim.merge(df[["GEO_ID", col_c01_total]], on="GEO_ID", how="left")

# Renaming the column and changing to numeric
acs_trim[col_c01_total] = pd.to_numeric(acs_trim[col_c01_total], errors="coerce")
acs_trim = acs_trim.rename(columns={col_c01_total: "S1903_C01_001E | Households (count)"})

In [ ]:
# Reviewing the table as is
acs_trim.head(5)

,GEO_ID,NAME,S1903_C02_002E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!White,S1903_C02_003E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Black or African American,S1903_C02_004E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!American Indian and Alaska Native,S1903_C02_005E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Asian,S1903_C02_006E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Native Hawaiian and Other Pacific Islander,S1903_C02_007E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Some other race,S1903_C02_008E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!Two or more races,S1903_C02_009E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!Hispanic or Latino origin (of any race),...,S1903_C03_032E | Estimate!!Median income (dollars)!!FAMILY INCOME BY NUMBER OF EARNERS!!2 earners,S1903_C03_033E | Estimate!!Median income (dollars)!!FAMILY INCOME BY NUMBER OF EARNERS!!3 or more earners,S1903_C03_034E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households,S1903_C03_035E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder,S1903_C03_036E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder!!Living alone,S1903_C03_037E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder!!Not living alone,S1903_C03_038E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder,S1903_C03_039E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder!!Living alone,S1903_C03_040E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder!!Not living alone,S1903_C01_001E | Households (count)
0,0500000US01001,"Autauga County, Alabama",75.0,18.6,0.1,1.0,0.0,1.0,4.2,3.7,...,103776.0,151521.0,38506.0,33293.0,32570.0,45509.0,48909.0,43900.0,88750.0,22523
1,0500000US01003,"Baldwin County, Alabama",85.9,8.1,0.3,0.6,0.0,1.8,3.3,3.9,...,118730.0,142571.0,41784.0,34592.0,32996.0,66490.0,50819.0,42586.0,85794.0,94642
2,0500000US01005,"Barbour County, Alabama",47.9,45.1,0.2,0.6,0.0,4.2,2.0,4.2,...,88924.0,99850.0,23054.0,23114.0,22011.0,49583.0,22796.0,22566.0,NaN,9080
3,0500000US01007,"Bibb County, Alabama",78.3,19.3,1.1,0.0,0.0,0.1,1.1,2.2,...,93508.0,119625.0,25288.0,21230.0,19528.0,93162.0,28048.0,26750.0,NaN,7571
4,0500000US01009,"Blount County, Alabama",92.0,0.7,0.3,0.1,0.0,2.0,4.8,7.6,...,96604.0,111322.0,28006.0,21286.0,19949.0,46449.0,38735.0,29075.0,81250.0,21977


In [ ]:
# Moving C01_001E to the front of the table

id_cols = ["GEO_ID", "NAME"]
count_col = "S1903_C01_001E | Households (count)"
new_order = id_cols + [count_col] + [c for c in acs_trim.columns if c not in id_cols + [count_col]]

acs_trim = acs_trim[new_order]

print(acs_trim.columns[:10])
acs_trim.head(2)

Index(['GEO_ID', 'NAME', 'S1903_C01_001E | Households (count)',
       'S1903_C02_002E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!White',
       'S1903_C02_003E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Black or African American',
       'S1903_C02_004E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!American Indian and Alaska Native',
       'S1903_C02_005E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Asian',
       'S1903_C02_006E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Native Hawaiian and Other Pacific Islander',
       'S1903_C02_007E | Estimate!!Percent Distribu

,GEO_ID,NAME,S1903_C01_001E | Households (count),S1903_C02_002E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!White,S1903_C02_003E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Black or African American,S1903_C02_004E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!American Indian and Alaska Native,S1903_C02_005E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Asian,S1903_C02_006E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Native Hawaiian and Other Pacific Islander,S1903_C02_007E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Some other race,S1903_C02_008E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!Two or more races,...,S1903_C03_031E | Estimate!!Median income (dollars)!!FAMILY INCOME BY NUMBER OF EARNERS!!1 earner,S1903_C03_032E | Estimate!!Median income (dollars)!!FAMILY INCOME BY NUMBER OF EARNERS!!2 earners,S1903_C03_033E | Estimate!!Median income (dollars)!!FAMILY INCOME BY NUMBER OF EARNERS!!3 or more earners,S1903_C03_034E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households,S1903_C03_035E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder,S1903_C03_036E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder!!Living alone,S1903_C03_037E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Female householder!!Not living alone,S1903_C03_038E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder,S1903_C03_039E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder!!Living alone,S1903_C03_040E | Estimate!!Median income (dollars)!!NONFAMILY HOUSEHOLDS!!Nonfamily households!!Male householder!!Not living alone
0,0500000US01001,"Autauga County, Alabama",22523,75.0,18.6,0.1,1.0,0.0,1.0,4.2,...,61202.0,103776.0,151521.0,38506.0,33293.0,32570.0,45509.0,48909.0,43900.0,88750.0
1,0500000US01003,"Baldwin County, Alabama",94642,85.9,8.1,0.3,0.6,0.0,1.8,3.3,...,67157.0,118730.0,142571.0,41784.0,34592.0,32996.0,66490.0,50819.0,42586.0,85794.0


In [ ]:
# Viewing all the column headers
print(acs_trim.columns[:82])

Index(['GEO_ID', 'NAME', 'S1903_C01_001E | Households (count)',
       'S1903_C02_002E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!White',
       'S1903_C02_003E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Black or African American',
       'S1903_C02_004E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!American Indian and Alaska Native',
       'S1903_C02_005E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Asian',
       'S1903_C02_006E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Native Hawaiian and Other Pacific Islander',
       'S1903_C02_007E | Estimate!!Percent Distribu

In [ ]:
# Converting headers to short snake_head labels
import re

def code_of(col: str) -> str:
    # "S1903_C02_002E | Estimate!!Percent Distribution!!...!!White alone" -> "S1903_C02_002E"
    return col.split(" | ", 1)[0]

def last_segments(label: str, n=2) -> str:
    # keep the last n pieces after "!!" (usually the most informative)
    parts = [p.strip() for p in str(label).split("!!") if p]
    return " ".join(parts[-n:]) if parts else label

def slug(s: str) -> str:
    s = s.lower()
    s = s.replace("&", "and")
    s = re.sub(r"[^a-z0-9]+", "_", s)     # non-alnum -> underscore
    s = re.sub(r"_+", "_", s).strip("_")  # collapse repeats
    return s

In [ ]:
# Building the rename map
rename_map = {}

for col in acs_trim.columns:
    if col in ("GEO_ID", "NAME"):
        continue

    code = code_of(col)  # e.g., S1903_C02_002E
    # Special case: the count we added
    if code == "S1903_C01_001E":
        rename_map[col] = "households_count"
        continue

    # Pull the label part after " | " if present
    label_part = col.split(" | ", 1)[1] if " | " in col else ""
    # Keep the last two segments (e.g., "Nonfamily households!!Male householder" -> "Male householder")
    short = last_segments(label_part, n=2)

    # Decide prefix by code family
    if "_C02_" in code and code.endswith("E"):
        # Percent distribution
        name = f"pct_{slug(short)}"
    elif "_C03_" in code and code.endswith("E"):
        # Median income (dollars)
        name = f"median_{slug(short)}_usd"
    else:
        # Fallback if something else slipped through
        name = slug(short) or code.lower()

    # Ensure uniqueness: if duplicate, append the ACS code
    if name in rename_map.values():
        name = f"{name}__{code.lower()}"

    rename_map[col] = name

# Quick peek at a few mappings
list(rename_map.items())[:8]

[('S1903_C01_001E | Households (count)', 'households_count'),
 ('S1903_C02_002E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!White',
  'pct_one_race_white'),
 ('S1903_C02_003E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Black or African American',
  'pct_one_race_black_or_african_american'),
 ('S1903_C02_004E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!American Indian and Alaska Native',
  'pct_one_race_american_indian_and_alaska_native'),
 ('S1903_C02_005E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!!Households!!One race--!!Asian',
  'pct_one_race_asian'),
 ('S1903_C02_006E | Estimate!!Percent Distribution!!HOUSEHOLD INCOME BY RACE AND HISPANIC OR LATINO ORIGIN OF HOUSEHOLDER!

In [ ]:
# Applying the rename
acs_clean = acs_trim.rename(columns=rename_map).copy()

# confirm first 12 columns
acs_clean.columns[:12]
acs_clean.head(2)

,GEO_ID,NAME,households_count,pct_one_race_white,pct_one_race_black_or_african_american,pct_one_race_american_indian_and_alaska_native,pct_one_race_asian,pct_one_race_native_hawaiian_and_other_pacific_islander,pct_one_race_some_other_race,pct_households_two_or_more_races,...,median_family_income_by_number_of_earners_1_earner_usd,median_family_income_by_number_of_earners_2_earners_usd,median_family_income_by_number_of_earners_3_or_more_earners_usd,median_nonfamily_households_nonfamily_households_usd,median_nonfamily_households_female_householder_usd,median_female_householder_living_alone_usd,median_female_householder_not_living_alone_usd,median_nonfamily_households_male_householder_usd,median_male_householder_living_alone_usd,median_male_householder_not_living_alone_usd
0,0500000US01001,"Autauga County, Alabama",22523,75.0,18.6,0.1,1.0,0.0,1.0,4.2,...,61202.0,103776.0,151521.0,38506.0,33293.0,32570.0,45509.0,48909.0,43900.0,88750.0
1,0500000US01003,"Baldwin County, Alabama",94642,85.9,8.1,0.3,0.6,0.0,1.8,3.3,...,67157.0,118730.0,142571.0,41784.0,34592.0,32996.0,66490.0,50819.0,42586.0,85794.0


In [ ]:
# Splitting county from state in ACS NAME column
acs_clean = acs_trim.rename(columns=rename_map).copy()

# Extract county_fips from GEO_ID (last 5 characters of GEO_ID string are the county FIPS)
acs_clean["county_fips"] = acs_clean["GEO_ID"].str[-5:]

# Split NAME into county_name and state_name
acs_clean[["county_name", "state_name"]] = acs_clean["NAME"].str.rsplit(",", n=1, expand=True)
acs_clean["county_name"] = (
    acs_clean["county_name"]
    .str.replace(" County", "", regex=False)
    .str.strip()
)
acs_clean["state_name"] = acs_clean["state_name"].str.strip()

# Reorder so identifiers show up first
id_cols = ["county_fips", "state_name", "county_name"]
other_cols = [c for c in acs_clean.columns if c not in id_cols and c not in ["GEO_ID","NAME"]]
acs_clean = acs_clean[id_cols + other_cols]

# Confirm
print(acs_clean.head(2))

# Save to Outputs
acs_clean.to_csv(OUT / "acs_county_2023_clean.csv", index=False)




  county_fips state_name county_name  households_count  pct_one_race_white  \
0       01001    Alabama     Autauga             22523                75.0   
1       01003    Alabama     Baldwin             94642                85.9   

   pct_one_race_black_or_african_american  \
0                                    18.6   
1                                     8.1   

   pct_one_race_american_indian_and_alaska_native  pct_one_race_asian  \
0                                             0.1                 1.0   
1                                             0.3                 0.6   

   pct_one_race_native_hawaiian_and_other_pacific_islander  \
0                                                0.0         
1                                                0.0         

   pct_one_race_some_other_race  ...  \
0                           1.0  ...   
1                           1.8  ...   

   median_family_income_by_number_of_earners_1_earner_usd  \
0                                       

In [ ]:
acs_clean.head(3)

,county_fips,state_name,county_name,households_count,pct_one_race_white,pct_one_race_black_or_african_american,pct_one_race_american_indian_and_alaska_native,pct_one_race_asian,pct_one_race_native_hawaiian_and_other_pacific_islander,pct_one_race_some_other_race,...,median_family_income_by_number_of_earners_1_earner_usd,median_family_income_by_number_of_earners_2_earners_usd,median_family_income_by_number_of_earners_3_or_more_earners_usd,median_nonfamily_households_nonfamily_households_usd,median_nonfamily_households_female_householder_usd,median_female_householder_living_alone_usd,median_female_householder_not_living_alone_usd,median_nonfamily_households_male_householder_usd,median_male_householder_living_alone_usd,median_male_householder_not_living_alone_usd
0,01001,Alabama,Autauga,22523,75.0,18.6,0.1,1.0,0.0,1.0,...,61202.0,103776.0,151521.0,38506.0,33293.0,32570.0,45509.0,48909.0,43900.0,88750.0
1,01003,Alabama,Baldwin,94642,85.9,8.1,0.3,0.6,0.0,1.8,...,67157.0,118730.0,142571.0,41784.0,34592.0,32996.0,66490.0,50819.0,42586.0,85794.0
2,01005,Alabama,Barbour,9080,47.9,45.1,0.2,0.6,0.0,4.2,...,44600.0,88924.0,99850.0,23054.0,23114.0,22011.0,49583.0,22796.0,22566.0,NaN


MOVING ON TO RUCC ^_^

In [ ]:
# Mount my google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Access files from my google drive
from pathlib import Path
BASE = Path("/content/drive/MyDrive/GRADTDA5621_Project_I") # THIS IS THE PATH TO THE FOLDER WITH THE FILES
RAW  = BASE / "Data_Sources" # THIS IS WHERE MY RAW FILES LIVE
RAW


Mounted at /content/drive


PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources')

In [ ]:
# List all files in my folder
list(RAW.glob("*"))

[PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/nanda_csv_readme.txt'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/nanda_grocery_Tract10_1990-2021_01P.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/nanda_grocery_ZCTA10_1990-2021_01P.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/nanda_grocery_Tract20_1990-2021_01P.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/nanda_grocery_ZCTA20_1990-2021_01P.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/Ruralurbancontinuumcodes2023.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/ACSST5Y2023.S1903-Column-Metadata.csv'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/ACSST5Y2023.S1903-Table-Notes.txt'),
 PosixPath('/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/ACSST5Y2023.S1903-Data.csv')]

I was running into some issues in loading and reading the RUCC file. Turns out that RUCC isn't in UTF-8 which is pandas default. Trying Windows-1252/Latin-1, then cp1252 if that does not work.

In [ ]:
# Telling pandas which encoding tyoe to read
rucc = pd.read_csv(rucc_path, encoding="latin1")

print(rucc.shape)
rucc.head(5)

(9703, 5)


,FIPS,State,County_Name,Attribute,Value
0,1001,AL,Autauga County,Population_2020,58805
1,1001,AL,Autauga County,RUCC_2023,2
2,1001,AL,Autauga County,Description,"Metro - Counties in metro areas of 250,000 to ..."
3,1003,AL,Baldwin County,Population_2020,231767
4,1003,AL,Baldwin County,RUCC_2023,3


In [ ]:
# Viewing headers
print(rucc.columns.tolist())
rucc.head()

['FIPS', 'State', 'County_Name', 'Attribute', 'Value']


,FIPS,State,County_Name,Attribute,Value
0,1001,AL,Autauga County,Population_2020,58805
1,1001,AL,Autauga County,RUCC_2023,2
2,1001,AL,Autauga County,Description,"Metro - Counties in metro areas of 250,000 to ..."
3,1003,AL,Baldwin County,Population_2020,231767
4,1003,AL,Baldwin County,RUCC_2023,3


In [ ]:
# Loading RUCC file
rucc_path = RAW / "Ruralurbancontinuumcodes2023.csv"
rucc = pd.read_csv(rucc_path, dtype=str, encoding="latin1")

print(rucc.shape)
rucc.head(3)

(9703, 5)


,FIPS,State,County_Name,Attribute,Value
0,01001,AL,Autauga County,Population_2020,58805
1,01001,AL,Autauga County,RUCC_2023,2
2,01001,AL,Autauga County,Description,"Metro - Counties in metro areas of 250,000 to ..."


In [ ]:
# Confirming headers
print(rucc.columns.tolist())

expected = {"FIPS","State","County_Name","Attribute","Value"}
print("Missing:", expected - set(rucc.columns))
print("Extra:",   set(rucc.columns) - expected)

['FIPS', 'State', 'County_Name', 'Attribute', 'Value']
Missing: set()
Extra: set()


In [ ]:
# Normalize FIPS
# Keeping digits only and padding to 5
rucc["FIPS"] = rucc["FIPS"].str.extract(r"(\d+)", expand=False).fillna("").str.zfill(5)

# Keeping only rows that look like valid county FIPS
rucc = rucc[rucc["FIPS"].str.len() == 5].copy()

# Checking for my own barely existing sanity
rucc.head(3)

,FIPS,State,County_Name,Attribute,Value
0,01001,AL,Autauga County,Population_2020,58805
1,01001,AL,Autauga County,RUCC_2023,2
2,01001,AL,Autauga County,Description,"Metro - Counties in metro areas of 250,000 to ..."


In [ ]:
# Standardizing to snake_case
rucc = rucc.rename(columns={
    "State": "state_abbr",
    "County_Name": "county_name",
    "Attribute": "attribute",
    "Value": "value",
})
rucc.columns = [c.strip() for c in rucc.columns]

rucc.head(3)

,FIPS,state_abbr,county_name,attribute,value
0,01001,AL,Autauga County,Population_2020,58805
1,01001,AL,Autauga County,RUCC_2023,2
2,01001,AL,Autauga County,Description,"Metro - Counties in metro areas of 250,000 to ..."


In [ ]:
# Pivot rows so there is only one per county
rucc_w = (
    rucc.pivot_table(
        index=["FIPS","state_abbr","county_name"],
        columns="attribute",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)
rucc_w.columns.name = None
print(rucc_w.shape)
rucc_w.head(3)

(3235, 6)


,FIPS,state_abbr,county_name,Description,Population_2020,RUCC_2023
0,01001,AL,Autauga County,"Metro - Counties in metro areas of 250,000 to ...",58805,2
1,01003,AL,Baldwin County,Metro - Counties in metro areas of fewer than ...,231767,3
2,01005,AL,Barbour County,"Nonmetro - Urban population of 5,000 to 20,000...",25223,6


In [ ]:
# Standardzing again to snake_case
rucc_w = rucc_w.rename(columns={
    "RUCC_2023": "rucc_code",
    "Description": "rucc_desc",
    "Population_2020": "pop_2020",
})
rucc_w.head(3)

,FIPS,state_abbr,county_name,rucc_desc,pop_2020,rucc_code
0,01001,AL,Autauga County,"Metro - Counties in metro areas of 250,000 to ...",58805,2
1,01003,AL,Baldwin County,Metro - Counties in metro areas of fewer than ...,231767,3
2,01005,AL,Barbour County,"Nonmetro - Urban population of 5,000 to 20,000...",25223,6


In [ ]:
# Change type cast to numeric wherever appropriate
rucc_w["rucc_code"] = pd.to_numeric(rucc_w.get("rucc_code"), errors="coerce").astype("Int64")
rucc_w["pop_2020"]  = pd.to_numeric(rucc_w.get("pop_2020"),  errors="coerce")
rucc_w.dtypes.head(10)

,0
FIPS,object
state_abbr,object
county_name,object
rucc_desc,object
pop_2020,int64
rucc_code,Int64


In [ ]:
# Adding metro and non-metro flags
rucc_w["rucc_metro_flag"] = pd.NA
mask = rucc_w["rucc_code"].notna()
rucc_w.loc[mask, "rucc_metro_flag"] = (
    rucc_w.loc[mask, "rucc_code"].astype(int).between(1,3)
    .map({True: "Metro", False: "Nonmetro"})
)

rucc_w[["county_name","state_abbr","rucc_code","rucc_desc","rucc_metro_flag"]].head(10)

,county_name,state_abbr,rucc_code,rucc_desc,rucc_metro_flag
0,Autauga County,AL,2,"Metro - Counties in metro areas of 250,000 to ...",Metro
1,Baldwin County,AL,3,Metro - Counties in metro areas of fewer than ...,Metro
2,Barbour County,AL,6,"Nonmetro - Urban population of 5,000 to 20,000...",Nonmetro
3,Bibb County,AL,1,Metro - Counties in metro areas of 1 million p...,Metro
4,Blount County,AL,1,Metro - Counties in metro areas of 1 million p...,Metro
5,Bullock County,AL,8,"Nonmetro - Urban population of fewer than 5,00...",Nonmetro
6,Butler County,AL,6,"Nonmetro - Urban population of 5,000 to 20,000...",Nonmetro
7,Calhoun County,AL,3,Metro - Counties in metro areas of fewer than ...,Metro
8,Chambers County,AL,6,"Nonmetro - Urban population of 5,000 to 20,000...",Nonmetro
9,Cherokee County,AL,8,"Nonmetro - Urban population of fewer than 5,00...",Nonmetro


In [ ]:
rucc_w.head(5)

,FIPS,state_abbr,county_name,rucc_desc,pop_2020,rucc_code,rucc_metro_flag
0,01001,AL,Autauga County,"Metro - Counties in metro areas of 250,000 to ...",58805,2,Metro
1,01003,AL,Baldwin County,Metro - Counties in metro areas of fewer than ...,231767,3,Metro
2,01005,AL,Barbour County,"Nonmetro - Urban population of 5,000 to 20,000...",25223,6,Nonmetro
3,01007,AL,Bibb County,Metro - Counties in metro areas of 1 million p...,22293,1,Metro
4,01009,AL,Blount County,Metro - Counties in metro areas of 1 million p...,59134,1,Metro


In [ ]:
# Checking for duplicates
dups = rucc_w["FIPS"].duplicated().sum()
print("Duplicate FIPS:", dups)
assert dups == 0, "Duplicate county FIPS found."

# Checking RUCC distribution and any missing values
print(rucc_w["rucc_code"].value_counts(dropna=False).sort_index())
print("Missing rucc_code:", rucc_w["rucc_code"].isna().sum())


Duplicate FIPS: 0
rucc_code
1       483
2       398
3       371
4       204
5        81
6       385
7       248
8       468
9       595
<NA>      2
Name: count, dtype: Int64
Missing rucc_code: 2


In [ ]:
# Checking which are the ones with missing RUCC
missing_rucc = rucc_w[rucc_w["rucc_code"].isna()].copy()

print(missing_rucc[["FIPS","state_abbr","county_name","rucc_desc"]])

# Checking by state
print(missing_rucc.groupby("state_abbr").size())

       FIPS state_abbr    county_name       rucc_desc
3146  60030         AS    Rose Island  Not Applicable
3147  60040         AS  Swains Island  Not Applicable
state_abbr
AS    2
dtype: int64


Looks like it is the American Samoa which are territories and not US counties so RUCC does not apply. Will drop these.

In [ ]:
from pathlib import Path

OUT = Path("Outputs")
OUT.mkdir(exist_ok=True)

# Aligning keys for ACS merge later and saving file
rucc_final = rucc_w.rename(columns={"FIPS":"county_fips"}).copy()

cols_order = [
    "county_fips","state_abbr","county_name",
    "rucc_code","rucc_metro_flag","rucc_desc","pop_2020"
]
rucc_final = rucc_final[cols_order + [c for c in rucc_final.columns if c not in cols_order]]

out_path = OUT / "rucc_2023_clean.csv"
rucc_final.to_csv(out_path, index=False)
print(f"Saved: {out_path} | Rows: {len(rucc_final):,}")

Saved: Outputs/rucc_2023_clean.csv | Rows: 3,235


In [ ]:
# Dropping the non-RUCC territories
rucc_final = rucc_final[rucc_final["rucc_code"].notna()].copy()

I couldn't find the new rucc file in my outputs folder so I reoriented myself and saved file in the proper folder.

In [ ]:
# Checking import path
import os
from pathlib import Path

print("CWD:", os.getcwd())
try:
    print("OUT:", OUT)
except NameError:
    print("OUT is not defined")

CWD: /content
OUT: Outputs


In [ ]:
# Checking where the new rucc file is
import glob
matches = glob.glob("**/rucc_2023_clean.csv", recursive=True)
print("Found files:", matches)

Found files: ['drive/MyDrive/Datasets/rucc_2023_clean.csv', 'Outputs/rucc_2023_clean.csv']


In [ ]:
# Remounting to save new rucc file in proper place
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Defining outputs folder
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Saving new rucc file in outputs folder
out_path = OUT / "rucc_2023_clean.csv"
rucc_final.to_csv(out_path, index=False)

print("Saved to:", out_path.resolve())

Saved to: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/rucc_2023_clean.csv


In [ ]:
# Confirming
import os
print(os.listdir(OUT))

['acs_county_2023_clean.csv', 'rucc_2023_clean.csv']


Now that both acs and rucc files are in the same outputs folder, I will perform some QA checks before I merge both files together.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Obtaining files for QA checks
OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

acs_path  = OUT / "acs_county_2023_clean.csv"
rucc_path = OUT / "rucc_2023_clean.csv"

acs  = pd.read_csv(acs_path, dtype=str)
rucc = pd.read_csv(rucc_path, dtype=str)

# Ensuring normalized keys
acs["county_fips"]  = acs["county_fips"].str.strip().str.zfill(5)
rucc["county_fips"] = rucc["county_fips"].str.strip().str.zfill(5)

In [ ]:
# Defining a QC function to later apply to ACS and RUCC file
def qc_basic(df, name, key="county_fips"):
    print(f"\n=== QC: {name} ===")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("First 10 columns:", df.columns[:10].tolist())

    if key not in df.columns:
        raise ValueError(f"{name}: missing key column '{key}'")

    # FIPS formatting
    bad_len = df[~df[key].str.match(r"^\d{5}$", na=False)]
    print("Bad FIPS format count:", len(bad_len))

    # duplicates
    dup_count = df.duplicated(subset=[key]).sum()
    print("Duplicate keys:", dup_count)

    # nulls
    null_keys = df[key].isna().sum()
    print("Null keys:", null_keys)

    if len(bad_len) or dup_count or null_keys:
        print("\nExamples of problem rows:")
        display(df[df[key].isna() | df.duplicated(subset=[key]) | (~df[key].str.match(r'^\d{5}$', na=False))].head(10))

In [ ]:
# Applying the QC function to the files
qc_basic(acs,  "ACS 2023")
qc_basic(rucc, "RUCC 2023")


=== QC: ACS 2023 ===
Rows: 3222
Columns: 83
First 10 columns: ['county_fips', 'state_name', 'county_name', 'households_count', 'pct_one_race_white', 'pct_one_race_black_or_african_american', 'pct_one_race_american_indian_and_alaska_native', 'pct_one_race_asian', 'pct_one_race_native_hawaiian_and_other_pacific_islander', 'pct_one_race_some_other_race']
Bad FIPS format count: 0
Duplicate keys: 0
Null keys: 0

=== QC: RUCC 2023 ===
Rows: 3235
Columns: 7
First 10 columns: ['county_fips', 'state_abbr', 'county_name', 'rucc_code', 'rucc_metro_flag', 'rucc_desc', 'pop_2020']
Bad FIPS format count: 0
Duplicate keys: 0
Null keys: 0


In [ ]:
# RUCC SPECIFIC QC
# Ensure rucc_code is numeric nullable Int64
rucc["rucc_code"] = pd.to_numeric(rucc.get("rucc_code"), errors="coerce").astype("Int64")

print("\nRUCC code distribution:")
print(rucc["rucc_code"].value_counts(dropna=False).sort_index())

# Drop territories with "NA"
rucc_keep = rucc[rucc["rucc_code"].notna()].copy()
print("RUCC rows after dropping NA rucc_code:", len(rucc_keep))



RUCC code distribution:
rucc_code
1       483
2       398
3       371
4       204
5        81
6       385
7       248
8       468
9       595
<NA>      2
Name: count, dtype: Int64
RUCC rows after dropping NA rucc_code: 3233


In [ ]:
# ACS SPECIFIC QC
# Ensuring one row per county_FIPS
dups_acs = acs.duplicated(subset=["county_fips"]).sum()
print("\nACS duplicate county_fips rows:", dups_acs)

# Cast commonly-numeric columns
num_cols = [c for c in acs.columns if c.startswith(("pop_", "median_", "pct_", "households_", "land_area"))]
for c in num_cols:
    acs[c] = pd.to_numeric(acs[c], errors="coerce")

# Just ensuring certain columns do not appear
if "pop_total" in acs.columns:
    print("ACS rows with pop_total <= 0:", int((acs["pop_total"] <= 0).fillna(False).sum()))

if "land_area_sqmi" in acs.columns:
    print("ACS rows with land_area_sqmi <= 0:", int((acs["land_area_sqmi"] <= 0).fillna(False).sum()))



ACS duplicate county_fips rows: 0


In [ ]:
# Cross dataset QC
only_in_acs  = sorted(set(acs["county_fips"])  - set(rucc_keep["county_fips"]))
only_in_rucc = sorted(set(rucc_keep["county_fips"]) - set(acs["county_fips"]))

print("\nOnly in ACS (not in RUCC):", len(only_in_acs))
print("Only in RUCC (not in ACS):", len(only_in_rucc))

# Saving small audit lists just in case they are needed later on
import pandas as pd
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
pd.DataFrame({"county_fips": only_in_acs}).to_csv(OUT / "audit_only_in_acs.csv", index=False)
pd.DataFrame({"county_fips": only_in_rucc}).to_csv(OUT / "audit_only_in_rucc.csv", index=False)

# Peek examples if any exist
if len(only_in_acs):
    print("\nSample ACS rows not in RUCC:")
    display(acs[acs["county_fips"].isin(only_in_acs)].head(10))

if len(only_in_rucc):
    print("\nSample RUCC rows not in ACS:")
    display(rucc_keep[rucc_keep["county_fips"].isin(only_in_rucc)][
        ["county_fips","state_abbr","county_name","rucc_code","rucc_metro_flag"]
    ].head(10))



Only in ACS (not in RUCC): 0
Only in RUCC (not in ACS): 11

Sample RUCC rows not in ACS:


,county_fips,state_abbr,county_name,rucc_code,rucc_metro_flag
3144,60010,AS,Eastern District,7,Nonmetro
3145,60020,AS,Manu'a District,9,Nonmetro
3148,60050,AS,Western District,5,Nonmetro
3149,66010,GU,Guam,5,Nonmetro
3150,69085,MP,Northern Islands Municipality,9,Nonmetro
3151,69100,MP,Rota Municipality,9,Nonmetro
3152,69110,MP,Saipan Municipality,5,Nonmetro
3153,69120,MP,Tinian Municipality,9,Nonmetro
3232,78010,VI,St. Croix Island,5,Nonmetro
3233,78020,VI,St. John Island,9,Nonmetro


In [ ]:
# Filtering so RUCC only shows US states
import pandas as pd
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")

# Reload
acs  = pd.read_csv(OUT / "acs_county_2023_clean.csv", dtype=str)
rucc = pd.read_csv(OUT / "rucc_2023_clean.csv", dtype=str)

# Normalize keys
acs["county_fips"]  = acs["county_fips"].str.zfill(5)
rucc["county_fips"] = rucc["county_fips"].str.zfill(5)

# Territory state FIPS prefixes to drop: AS=60, GU=66, MP=69, PR=72, VI=78
territory_prefixes = {"60","66","69","72","78"}
rucc["state_fips2"] = rucc["county_fips"].str[:2]
rucc_keep = rucc[~rucc["state_fips2"].isin(territory_prefixes)].copy()

print("RUCC rows before:", len(rucc))
print("RUCC rows after dropping territories:", len(rucc_keep))

RUCC rows before: 3235
RUCC rows after dropping territories: 3144


In [ ]:
# Creating a dictionary for state only FIPS
valid_state_fips = {
    "01","02","04","05","06","08","09","10","11","12","13","15","16","17","18","19",
    "20","21","22","23","24","25","26","27","28","29","30","31","32","33","34","35",
    "36","37","38","39","40","41","42","44","45","46","47","48","49","50","51","53",
    "54","55","56"
}

In [ ]:
# Filtering ACS and RUCC for state only FIPS
acs["state_fips2"]  = acs["county_fips"].str[:2]
rucc["state_fips2"] = rucc["county_fips"].str[:2]

acs_keep  = acs[acs["state_fips2"].isin(valid_state_fips)].copy()
rucc_keep = rucc[rucc["state_fips2"].isin(valid_state_fips)].copy()

print("ACS before:", len(acs), "| after:", len(acs_keep))
print("RUCC before:", len(rucc), "| after:", len(rucc_keep))

ACS before: 3222 | after: 3144
RUCC before: 3235 | after: 3144


Now that ACS and RUCC have state only FIPS, we can merge ^_^

In [ ]:
# Merging yaaaaay
acs_rucc = acs_keep.merge(
    rucc_keep.drop(columns=["state_fips2"]),
    on="county_fips", how="inner", suffixes=("", "_rucc")
)

print(f"Merged rows: {len(acs_rucc):,}")
out_path = OUT / "acs_rucc_2023.csv"
acs_rucc.to_csv(out_path, index=False)
print("Saved:", out_path)

Merged rows: 3,144
Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023.csv


In [ ]:
# Another check to ensure seamless merge
only_in_acs  = sorted(set(acs_keep["county_fips"])  - set(rucc_keep["county_fips"]))
only_in_rucc = sorted(set(rucc_keep["county_fips"]) - set(acs_keep["county_fips"]))

print("Only in ACS (not in RUCC):", len(only_in_acs))
print("Only in RUCC (not in ACS):", len(only_in_rucc))

Only in ACS (not in RUCC): 0
Only in RUCC (not in ACS): 0


Now that the files are merged, I will run a post-merge QA check.

In [ ]:
import pandas as pd
from pathlib import Path

# Loading acs_rucc merge and defining
OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
merged_path = OUT / "acs_rucc_2023.csv"

try:
    acs_rucc
except NameError:
    acs_rucc = pd.read_csv(merged_path, dtype=str)

# Ensuring county_fips is always 5 characters and casting rucc_code to nullable integer
acs_rucc["county_fips"] = acs_rucc["county_fips"].str.zfill(5)
if "rucc_code" in acs_rucc.columns:
    acs_rucc["rucc_code"] = pd.to_numeric(acs_rucc["rucc_code"], errors="coerce").astype("Int64")

print("Rows:", len(acs_rucc), "| Unique counties:", acs_rucc["county_fips"].nunique())

Rows: 3144 | Unique counties: 3144


In [ ]:
# Checking duplicates and nulls
print("Duplicate county_fips:", acs_rucc.duplicated("county_fips").sum())
print("Null county_fips:", acs_rucc["county_fips"].isna().sum())

Duplicate county_fips: 0
Null county_fips: 0


In [ ]:
# Ensuring every county has a RUCC code
print("Missing rucc_code:", acs_rucc["rucc_code"].isna().sum())

print("RUCC distribution:")
print(acs_rucc["rucc_code"].value_counts(dropna=False).sort_index())

Missing rucc_code: 0
RUCC distribution:
rucc_code
1    443
2    385
3    358
4    201
5     76
6    379
7    247
8    466
9    589
Name: count, dtype: Int64


In [ ]:
# Checking the metro and nonmetro distribution
if "rucc_metro_flag" in acs_rucc.columns:
    print("\nCounts by Metro/Nonmetro:")
    print(acs_rucc["rucc_metro_flag"].value_counts(dropna=False))


Counts by Metro/Nonmetro:
rucc_metro_flag
Nonmetro    1958
Metro       1186
Name: count, dtype: int64


In [ ]:
# Checking state level counts
acs_rucc["state_fips2"] = acs_rucc["county_fips"].str[:2]
by_state_counts = acs_rucc.groupby("state_fips2").size().sort_index()
print("\nCounts by state FIPS:")
print(by_state_counts)


Counts by state FIPS:
state_fips2
01     67
02     30
04     15
05     75
06     58
08     64
09      9
10      3
11      1
12     67
13    159
15      5
16     44
17    102
18     92
19     99
20    105
21    120
22     64
23     16
24     24
25     14
26     83
27     87
28     82
29    115
30     56
31     93
32     17
33     10
34     21
35     33
36     62
37    100
38     53
39     88
40     77
41     36
42     67
44      5
45     46
46     66
47     95
48    254
49     29
50     14
51    133
53     39
54     55
55     72
56     23
dtype: int64


In [ ]:
acs_rucc.head(5)

,county_fips,state_name,county_name,households_count,pct_one_race_white,pct_one_race_black_or_african_american,pct_one_race_american_indian_and_alaska_native,pct_one_race_asian,pct_one_race_native_hawaiian_and_other_pacific_islander,pct_one_race_some_other_race,...,median_nonfamily_households_male_householder_usd,median_male_householder_living_alone_usd,median_male_householder_not_living_alone_usd,state_fips2,state_abbr,county_name_rucc,rucc_code,rucc_metro_flag,rucc_desc,pop_2020
0,01001,Alabama,Autauga,22523,75.0,18.6,0.1,1.0,0.0,1.0,...,48909.0,43900.0,88750.0,01,AL,Autauga County,2,Metro,"Metro - Counties in metro areas of 250,000 to ...",58805
1,01003,Alabama,Baldwin,94642,85.9,8.1,0.3,0.6,0.0,1.8,...,50819.0,42586.0,85794.0,01,AL,Baldwin County,3,Metro,Metro - Counties in metro areas of fewer than ...,231767
2,01005,Alabama,Barbour,9080,47.9,45.1,0.2,0.6,0.0,4.2,...,22796.0,22566.0,NaN,01,AL,Barbour County,6,Nonmetro,"Nonmetro - Urban population of 5,000 to 20,000...",25223
3,01007,Alabama,Bibb,7571,78.3,19.3,1.1,0.0,0.0,0.1,...,28048.0,26750.0,NaN,01,AL,Bibb County,1,Metro,Metro - Counties in metro areas of 1 million p...,22293
4,01009,Alabama,Blount,21977,92.0,0.7,0.3,0.1,0.0,2.0,...,38735.0,29075.0,81250.0,01,AL,Blount County,1,Metro,Metro - Counties in metro areas of 1 million p...,59134


In [ ]:
# Removing state_fips2 and resaving copy
acs_rucc.drop(columns=["state_fips2"], errors="ignore").to_csv(OUT / "acs_rucc_2023.csv", index=False)
print("Saved:", OUT / "acs_rucc_2023.csv")

Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023.csv


In [ ]:
# Baby EDA check just for fun
import pandas as pd
import numpy as np

acs_rucc = pd.read_csv("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023.csv", dtype=str)

# Inspecting columns so we know what we have
print("Columns:", list(acs_rucc.columns))
print("Rows:", len(acs_rucc), "Unique counties:", acs_rucc["county_fips"].nunique())

# Ensuring helpful flags exist and compute rucc_metro_flag if missing
if "rucc_code" in acs_rucc.columns and "rucc_metro_flag" not in acs_rucc.columns:
    rc = pd.to_numeric(acs_rucc["rucc_code"], errors="coerce")
    acs_rucc["rucc_metro_flag"] = np.where(rc.between(1,3), "Metro",
                                    np.where(rc.between(4,9), "Nonmetro", pd.NA))

# Auto-detect numeric columns and convert
candidate_num_prefixes = ("pop_", "median_", "pct_", "households_", "land_area", "emps_", "count_")
num_cols = [c for c in acs_rucc.columns if c.startswith(candidate_num_prefixes)]
for c in num_cols:
    acs_rucc[c] = pd.to_numeric(acs_rucc[c], errors="coerce")

print("\nDetected numeric columns:", num_cols[:12], "..." if len(num_cols) > 12 else "")

# Print a small numeric summary
common = [c for c in ["pop_total","median_income","households_count","land_area_sqmi"] if c in acs_rucc.columns]
summary_cols = common if common else num_cols[:6]  # fallback: first few numeric cols

if summary_cols:
    print("\nNumeric summary (selected columns):")
    display(acs_rucc[summary_cols].describe().T)
else:
    print("\n(No numeric columns detected to summarize.)")

# Group summary by Metro/Nonmetro if the flag exists
if "rucc_metro_flag" in acs_rucc.columns and summary_cols:
    print("\nGroup means by Metro/Nonmetro:")
    display(
        acs_rucc.groupby("rucc_metro_flag")[summary_cols]
                .mean(numeric_only=True)
                .round(2)
    )
else:
    print("\n(Metro/Nonmetro group summary skipped — missing 'rucc_metro_flag' or numeric columns.)")

# Quick top/bottom glance for a key metric if present
target_metric = None
for candidate in ["median_income","pop_total","households_count"]:
    if candidate in acs_rucc.columns:
        target_metric = candidate
        break

if target_metric:
    print(f"\nTop 5 counties by {target_metric}:")
    display(
        acs_rucc[["county_fips","state_name","county_name",target_metric]]
        .sort_values(target_metric, ascending=False)
        .head(5)
    )
    print(f"\nBottom 5 counties by {target_metric}:")
    display(
        acs_rucc[["county_fips","state_name","county_name",target_metric]]
        .sort_values(target_metric, ascending=True)
        .head(5)
    )
else:
    print("\n(No target metric like 'median_income' or 'pop_total' found for top/bottom listing.)")

Columns: ['county_fips', 'state_name', 'county_name', 'households_count', 'pct_one_race_white', 'pct_one_race_black_or_african_american', 'pct_one_race_american_indian_and_alaska_native', 'pct_one_race_asian', 'pct_one_race_native_hawaiian_and_other_pacific_islander', 'pct_one_race_some_other_race', 'pct_households_two_or_more_races', 'pct_households_hispanic_or_latino_origin_of_any_race', 'pct_households_white_alone_not_hispanic_or_latino', 'pct_household_income_by_age_of_householder_15_to_24_years', 'pct_household_income_by_age_of_householder_25_to_44_years', 'pct_household_income_by_age_of_householder_45_to_64_years', 'pct_household_income_by_age_of_householder_65_years_and_over', 'pct_families_families', 'pct_families_with_own_children_of_householder_under_18_years', 'pct_families_with_no_own_children_of_householder_under_18_years', 'pct_families_married_couple_families', 'pct_married_couple_families_with_own_children_under_18_years', 'pct_families_female_householder_no_spouse_pres

,count,mean,std,min,25%,50%,75%,max
households_count,3144.0,40547.985051,122958.128094,17.0,4201.0,10130.5,26760.75,3390254.0



Group means by Metro/Nonmetro:


,households_count
rucc_metro_flag,
Metro,92245.73
Nonmetro,9233.62



Top 5 counties by households_count:


,county_fips,state_name,county_name,households_count
205,06037,California,Los Angeles,3390254
612,17031,Illinois,Cook,2084578
2625,48201,Texas,Harris,1728103
104,04013,Arizona,Maricopa,1697342
223,06073,California,San Diego,1159822



Bottom 5 counties by households_count:


,county_fips,state_name,county_name,households_count
2655,48261,Texas,Kenedy,17
550,15005,Hawaii,Kalawao,22
2675,48301,Texas,Loving,36
2659,48269,Texas,King,90
2680,48311,Texas,McMullen,168


In [ ]:
# Cleaning up the table a bit more before NaNDA
# Check if any names are not the same
name_mismatch = (acs_rucc["county_name"].str.strip() != acs_rucc["county_name_rucc"].str.strip())
print("County name mismatches:", int(name_mismatch.sum()))

# If there are no mismatches, then drop county_name_rucc column
if name_mismatch.sum() == 0:
    acs_rucc = acs_rucc.drop(columns=["county_name_rucc"])

County name mismatches: 2999


Just noticed that one column includes suffixes. Will remove those and then double check for county name alignment again.

In [ ]:
import re
import unicodedata

def normalize_county_name(s: str) -> str:
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFKD", s).encode("ascii","ignore").decode("ascii").lower()
    s = s.strip()

    # Remove common suffixes
    suffixes = [
        " city and borough", " municipality", " census area", " consolidated government",
        " consolidated city", " city and county", " charter township", " charter county",
        " urban county", " independent city", " county", " parish", " borough", " city",
        " municipio"  # PR
    ]
    for suf in suffixes:
        if s.endswith(suf):
            s = s[: -len(suf)]

    # Collapse whitespace & remove punctuation
    s = re.sub(r"[\-'\.&]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Build normalized names for both columns
acs_rucc["name_norm_acs"]  = acs_rucc["county_name"].map(normalize_county_name)
acs_rucc["name_norm_rucc"] = acs_rucc["county_name_rucc"].map(normalize_county_name)

true_mismatch_mask = acs_rucc["name_norm_acs"] != acs_rucc["name_norm_rucc"]
print("Normalized name mismatches:", int(true_mismatch_mask.sum()))

# Inspect a few if any remain
acs_rucc.loc[true_mismatch_mask,
             ["county_fips","state_name","county_name","county_name_rucc","name_norm_acs","name_norm_rucc"]
            ].head(20)

Normalized name mismatches: 0


,county_fips,state_name,county_name,county_name_rucc,name_norm_acs,name_norm_rucc


In [ ]:
# Now only choosing one column
acs_rucc = acs_rucc.drop(columns=["county_name_rucc","name_norm_acs","name_norm_rucc"], errors="ignore")

In [ ]:
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

out_path = OUT / "acs_rucc_2023_clean.csv"
acs_rucc.to_csv(out_path, index=False)

print("Saved file:", out_path)
print("Rows:", len(acs_rucc), "| Columns:", len(acs_rucc.columns))

Saved file: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023_clean.csv
Rows: 3144 | Columns: 88


In [ ]:
test = pd.read_csv(out_path, dtype=str)
print("Reloaded rows:", len(test), "Columns:", len(test.columns))
test.head(3)

Reloaded rows: 3144 Columns: 88


,county_fips,state_name,county_name,households_count,pct_one_race_white,pct_one_race_black_or_african_american,pct_one_race_american_indian_and_alaska_native,pct_one_race_asian,pct_one_race_native_hawaiian_and_other_pacific_islander,pct_one_race_some_other_race,...,median_female_householder_living_alone_usd,median_female_householder_not_living_alone_usd,median_nonfamily_households_male_householder_usd,median_male_householder_living_alone_usd,median_male_householder_not_living_alone_usd,state_abbr,rucc_code,rucc_metro_flag,rucc_desc,pop_2020
0,01001,Alabama,Autauga,22523,75.0,18.6,0.1,1.0,0.0,1.0,...,32570.0,45509.0,48909.0,43900.0,88750.0,AL,2,Metro,"Metro - Counties in metro areas of 250,000 to ...",58805
1,01003,Alabama,Baldwin,94642,85.9,8.1,0.3,0.6,0.0,1.8,...,32996.0,66490.0,50819.0,42586.0,85794.0,AL,3,Metro,Metro - Counties in metro areas of fewer than ...,231767
2,01005,Alabama,Barbour,9080,47.9,45.1,0.2,0.6,0.0,4.2,...,22011.0,49583.0,22796.0,22566.0,NaN,AL,6,Nonmetro,"Nonmetro - Urban population of 5,000 to 20,000...",25223


In [ ]:
# Checking headers again
print(list(acs_rucc.columns))

['county_fips', 'state_name', 'county_name', 'households_count', 'pct_one_race_white', 'pct_one_race_black_or_african_american', 'pct_one_race_american_indian_and_alaska_native', 'pct_one_race_asian', 'pct_one_race_native_hawaiian_and_other_pacific_islander', 'pct_one_race_some_other_race', 'pct_households_two_or_more_races', 'pct_households_hispanic_or_latino_origin_of_any_race', 'pct_households_white_alone_not_hispanic_or_latino', 'pct_household_income_by_age_of_householder_15_to_24_years', 'pct_household_income_by_age_of_householder_25_to_44_years', 'pct_household_income_by_age_of_householder_45_to_64_years', 'pct_household_income_by_age_of_householder_65_years_and_over', 'pct_families_families', 'pct_families_with_own_children_of_householder_under_18_years', 'pct_families_with_no_own_children_of_householder_under_18_years', 'pct_families_married_couple_families', 'pct_married_couple_families_with_own_children_under_18_years', 'pct_families_female_householder_no_spouse_present', 'pc

In [ ]:
# Reorganizing columns just so the table flows a bit better
# Define column groups
id_cols = [
    "county_fips","state_name","state_abbr","county_name",
    "rucc_code","rucc_metro_flag","rucc_desc","pop_2020"
]

core_cols = [
    "households_count"
]

race_pct_cols = [
    "pct_one_race_white","pct_one_race_black_or_african_american",
    "pct_one_race_american_indian_and_alaska_native","pct_one_race_asian",
    "pct_one_race_native_hawaiian_and_other_pacific_islander",
    "pct_one_race_some_other_race","pct_households_two_or_more_races",
    "pct_households_hispanic_or_latino_origin_of_any_race",
    "pct_households_white_alone_not_hispanic_or_latino"
]

household_income_pct_cols = [
    "pct_household_income_by_age_of_householder_15_to_24_years",
    "pct_household_income_by_age_of_householder_25_to_44_years",
    "pct_household_income_by_age_of_householder_45_to_64_years",
    "pct_household_income_by_age_of_householder_65_years_and_over",
    "pct_family_income_by_family_size_2_person_families",
    "pct_family_income_by_family_size_3_person_families",
    "pct_family_income_by_family_size_4_person_families",
    "pct_family_income_by_family_size_5_person_families",
    "pct_family_income_by_family_size_6_person_families",
    "pct_family_income_by_family_size_7_or_more_person_families",
    "pct_family_income_by_number_of_earners_no_earners",
    "pct_family_income_by_number_of_earners_1_earner",
    "pct_family_income_by_number_of_earners_2_earners",
    "pct_family_income_by_number_of_earners_3_or_more_earners",
]

family_structure_pct_cols = [
    "pct_families_families","pct_families_with_own_children_of_householder_under_18_years",
    "pct_families_with_no_own_children_of_householder_under_18_years",
    "pct_families_married_couple_families","pct_married_couple_families_with_own_children_under_18_years",
    "pct_families_female_householder_no_spouse_present",
    "pct_female_householder_no_spouse_present_with_own_children_under_18_years",
    "pct_families_male_householder_no_spouse_present",
    "pct_male_householder_no_spouse_present_with_own_children_under_18_years",
    "pct_nonfamily_households_nonfamily_households",
    "pct_nonfamily_households_female_householder","pct_female_householder_living_alone",
    "pct_female_householder_not_living_alone","pct_nonfamily_households_male_householder",
    "pct_male_householder_living_alone","pct_male_householder_not_living_alone"
]

median_income_cols = [
    "median_household_income_by_race_and_hispanic_or_latino_origin_of_householder_households_usd",
    "median_one_race_white_usd","median_one_race_black_or_african_american_usd",
    "median_one_race_american_indian_and_alaska_native_usd","median_one_race_asian_usd",
    "median_one_race_native_hawaiian_and_other_pacific_islander_usd","median_one_race_some_other_race_usd",
    "median_households_two_or_more_races_usd","median_households_hispanic_or_latino_origin_of_any_race_usd",
    "median_households_white_alone_not_hispanic_or_latino_usd",
    "median_household_income_by_age_of_householder_15_to_24_years_usd",
    "median_household_income_by_age_of_householder_25_to_44_years_usd",
    "median_household_income_by_age_of_householder_45_to_64_years_usd",
    "median_household_income_by_age_of_householder_65_years_and_over_usd",
    "median_families_families_usd","median_families_with_own_children_of_householder_under_18_years_usd",
    "median_families_with_no_own_children_of_householder_under_18_years_usd",
    "median_families_married_couple_families_usd","median_married_couple_families_with_own_children_under_18_years_usd",
    "median_families_female_householder_no_spouse_present_usd",
    "median_female_householder_no_spouse_present_with_own_children_under_18_years_usd",
    "median_families_male_householder_no_spouse_present_usd",
    "median_male_householder_no_spouse_present_with_own_children_under_18_years_usd",
    "median_family_income_by_family_size_2_person_families_usd",
    "median_family_income_by_family_size_3_person_families_usd",
    "median_family_income_by_family_size_4_person_families_usd",
    "median_family_income_by_family_size_5_person_families_usd",
    "median_family_income_by_family_size_6_person_families_usd",
    "median_family_income_by_family_size_7_or_more_person_families_usd",
    "median_family_income_by_number_of_earners_no_earners_usd",
    "median_family_income_by_number_of_earners_1_earner_usd",
    "median_family_income_by_number_of_earners_2_earners_usd",
    "median_family_income_by_number_of_earners_3_or_more_earners_usd",
    "median_nonfamily_households_nonfamily_households_usd",
    "median_nonfamily_households_female_householder_usd","median_female_householder_living_alone_usd",
    "median_female_householder_not_living_alone_usd","median_nonfamily_households_male_householder_usd",
    "median_male_householder_living_alone_usd","median_male_householder_not_living_alone_usd"
]

# Reorder DataFrame
all_ordered = id_cols + core_cols + race_pct_cols + household_income_pct_cols + family_structure_pct_cols + median_income_cols
acs_rucc = acs_rucc[[c for c in all_ordered if c in acs_rucc.columns]]

# Preview the new order
print(acs_rucc.columns[:20])


Index(['county_fips', 'state_name', 'state_abbr', 'county_name', 'rucc_code',
       'rucc_metro_flag', 'rucc_desc', 'pop_2020', 'households_count',
       'pct_one_race_white', 'pct_one_race_black_or_african_american',
       'pct_one_race_american_indian_and_alaska_native', 'pct_one_race_asian',
       'pct_one_race_native_hawaiian_and_other_pacific_islander',
       'pct_one_race_some_other_race', 'pct_households_two_or_more_races',
       'pct_households_hispanic_or_latino_origin_of_any_race',
       'pct_households_white_alone_not_hispanic_or_latino',
       'pct_household_income_by_age_of_householder_15_to_24_years',
       'pct_household_income_by_age_of_householder_25_to_44_years'],
      dtype='object')


In [ ]:
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

# Save a new, clearly labeled version
out_path = OUT / "acs_rucc_2023_reordered.csv"
acs_rucc.to_csv(out_path, index=False)

print("Saved reordered file:", out_path)
print("Rows:", len(acs_rucc), "| Columns:", len(acs_rucc.columns))

Saved reordered file: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023_reordered.csv
Rows: 3144 | Columns: 88


In [ ]:
# Adding a README file
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

readme_path = OUT / "README_acs_rucc_2023.md"
readme_text = r"""<ACS + RUCC County Table (States + DC), 2023

What this file is
A county-level table for the 50 U.S. states + DC (≈3,144 rows), combining:
- ACS 2023 county features (cleaned & renamed)
- RUCC 2023 classification (USDA ERS)

Saved files:
- `acs_rucc_2023.csv` — merged baseline
- `acs_rucc_2023_reordered.csv` — same data, columns grouped for readability (recommended for EDA)

Join keys & scope
- Key: `county_fips` (5-digit, zero-padded)
- Universe: States + DC only (territories removed: AS, GU, MP, PR, VI)
- Year alignment: ACS (2019–2023 5-year) + RUCC 2023 (structural context)

Column groups

1) Identifiers
- `county_fips` — 5-digit FIPS (SSCCC)
- `state_name` — e.g., “Wisconsin”
- `state_abbr` — e.g., “WI”
- `county_name` — human-readable county/county-equivalent name
- `rucc_code` — 1–9 code (see below)
- `rucc_metro_flag` — `Metro` (RUCC 1–3) / `Nonmetro` (RUCC 4–9)
- `rucc_desc` — USDA ERS description of the RUCC class
- `pop_2020` — 2020 population (from RUCC file, for quick QA; use ACS pop for analysis if added later)

2) Core counts
- `households_count`
> (If later added: `pop_total`, `land_area_sqmi` go here.)

3) Race / Ethnicity (% of households or people; units = percent 0–100)
- `pct_one_race_white`, `pct_one_race_black_or_african_american`,
  `pct_one_race_american_indian_and_alaska_native`, `pct_one_race_asian`,
  `pct_one_race_native_hawaiian_and_other_pacific_islander`,
  `pct_one_race_some_other_race`, `pct_households_two_or_more_races`,
  `pct_households_hispanic_or_latino_origin_of_any_race`,
  `pct_households_white_alone_not_hispanic_or_latino`

4) Family / Household structure (%)
- `pct_families_families`, `pct_families_with_own_children_of_householder_under_18_years`,
  `pct_families_with_no_own_children_of_householder_under_18_years`,
  `pct_families_married_couple_families`,
  `pct_married_couple_families_with_own_children_under_18_years`,
  `pct_families_female_householder_no_spouse_present`,
  `pct_female_householder_no_spouse_present_with_own_children_under_18_years`,
  `pct_families_male_householder_no_spouse_present`,
  `pct_male_householder_no_spouse_present_with_own_children_under_18_years`,
  `pct_nonfamily_households_nonfamily_households`,
  `pct_nonfamily_households_female_householder`, `pct_female_householder_living_alone`,
  `pct_female_householder_not_living_alone`, `pct_nonfamily_households_male_householder`,
  `pct_male_householder_living_alone`, `pct_male_householder_not_living_alone`

5) Income composition (%)
- By age of householder:
  `pct_household_income_by_age_of_householder_15_to_24_years`,
  `..._25_to_44_years`, `..._45_to_64_years`, `..._65_years_and_over`
- By family size:
  `pct_family_income_by_family_size_2_person_families` … `..._7_or_more_person_families`
- By # of earners:
  `pct_family_income_by_number_of_earners_no_earners`,
  `..._1_earner`, `..._2_earners`, `..._3_or_more_earners`

6) Income medians ($ USD)
- By race/ethnicity:
  `median_household_income_by_race_and_hispanic_or_latino_origin_of_householder_households_usd`,
  `median_one_race_white_usd`, `median_one_race_black_or_african_american_usd`,
  `median_one_race_american_indian_and_alaska_native_usd`, `median_one_race_asian_usd`,
  `median_one_race_native_hawaiian_and_other_pacific_islander_usd`,
  `median_one_race_some_other_race_usd`, `median_households_two_or_more_races_usd`,
  `median_households_hispanic_or_latino_origin_of_any_race_usd`,
  `median_households_white_alone_not_hispanic_or_latino_usd`
- By age of householder:
  `median_household_income_by_age_of_householder_15_to_24_years_usd`,
  `..._25_to_44_years_usd`, `..._45_to_64_years_usd`, `..._65_years_and_over_usd`
- By family vs nonfamily:
  `median_families_families_usd`,
  `median_nonfamily_households_nonfamily_households_usd`,
  `median_nonfamily_households_female_householder_usd`,
  `median_nonfamily_households_male_householder_usd`,
  `median_female_householder_living_alone_usd`,
  `median_female_householder_not_living_alone_usd`,
  `median_male_householder_living_alone_usd`,
  `median_male_householder_not_living_alone_usd`
- By family size:
  `median_family_income_by_family_size_2_person_families_usd` … `..._7_or_more_person_families_usd`
- By # of earners:
  `median_family_income_by_number_of_earners_no_earners_usd`,
  `..._1_earner_usd`, `..._2_earners_usd`, `..._3_or_more_earners_usd`

RUCC code meanings (quick)
1 = Metro: ≥1 million
2 = Metro: 250k–<1M
3 = Metro: <250k
4 = Nonmetro: Urban (20k+) adjacent to metro
5 = Nonmetro: Urban (20k+) not adjacent
6 = Nonmetro: Urban (2.5k–19,999) adjacent
7 = Nonmetro: Urban (2.5k–19,999) not adjacent
8 = Nonmetro: Completely rural or <2.5k, adjacent
9 = Nonmetro: Completely rural or <2.5k, not adjacent

Data hygiene & assumptions
- `county_fips` is the authoritative key (not names).
- Percentages are 0–100; medians are USD.
- RUCC describes structure (not yearly), ACS measures are 2019–2023 5-year estimates.
- Territories excluded; states + DC only.

Known edge cases
- Independent cities (VA), consolidated city-counties (e.g., St. Louis city, Baltimore city), Alaska boroughs/census areas, and Kalawao (HI) are included as county-equivalents.

## Next step (optional)
Merge NaNDA Tract20 (year=2021) aggregated to county_fips; suffix new features with `_2021` and compute per-10k using ACS population when added.
>"""

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_text)

print("Saved:", readme_path)

Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/README_acs_rucc_2023.md


Decided to add more variable for a more comprehensive EDA. ChatGPT took the wheel on this section because I was too tired to do it manually step by step.

In [ ]:
# Making new variable table and merging to current rucc_acs
import pandas as pd
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

# Variable map (ACS 2023 5-year Data Profile + one detailed table)
vars_map = {
    "DP05_0001E":  "pop_total",                      # Total population
    "DP03_0062E":  "median_household_income_usd",    # Median household income ($)
    "DP03_0088E":  "per_capita_income_usd",          # Per-capita income ($)
    "DP03_0009PE": "unemployment_rate_pct",          # Unemployment rate (%)
    "DP02_0067PE": "pct_hs_or_higher",               # HS grad or higher (%), age 25+
    "DP02_0068PE": "pct_bachelors_or_higher",        # Bachelor's degree or higher (%), age 25+
    "DP02_0154PE": "pct_households_broadband",       # Households with broadband (%)
    "DP04_0058PE": "pct_households_no_vehicle",      # Households with no vehicle (%)
    "DP03_0074PE": "pct_households_snap",            # Households receiving SNAP (%)
    "DP03_0025E":  "mean_travel_time_to_work_min",   # Mean travel time to work (minutes)
}

# Pull Data Profile vars
base = "https://api.census.gov/data/2023/acs/acs5/profile"
get_vars = ",".join(["NAME"] + list(vars_map.keys()))
url = f"{base}?get={get_vars}&for=county:*&in=state:*"

acs_ctrl = pd.read_json(url)
acs_ctrl.columns = acs_ctrl.iloc[0]           # first row is header
acs_ctrl = acs_ctrl.drop(index=0).rename(columns=vars_map)

# Build 5-digit county FIPS
acs_ctrl["state"] = acs_ctrl["state"].str.zfill(2)
acs_ctrl["county"] = acs_ctrl["county"].str.zfill(3)
acs_ctrl["county_fips"] = acs_ctrl["state"] + acs_ctrl["county"]
acs_ctrl = acs_ctrl.drop(columns=["NAME","state","county"])

# Cast numerics
num_cols = list(vars_map.values())
acs_ctrl[num_cols] = acs_ctrl[num_cols].apply(pd.to_numeric, errors="coerce")

# Add Gini index from detailed table B19083_001E
base_det = "https://api.census.gov/data/2023/acs/acs5"
gini_url = f"{base_det}?get=B19083_001E,NAME&for=county:*&in=state:*"
gini = pd.read_json(gini_url)
gini.columns = gini.iloc[0]
gini = gini.drop(index=0).reset_index(drop=True)
gini = gini.rename(columns={"B19083_001E": "gini_index"})
gini["state"] = gini["state"].str.zfill(2)
gini["county"] = gini["county"].str.zfill(3)
gini["county_fips"] = gini["state"] + gini["county"]
gini["gini_index"] = pd.to_numeric(gini["gini_index"], errors="coerce")
gini = gini[["county_fips","gini_index"]]

acs_ctrl = acs_ctrl.merge(gini, on="county_fips", how="left")

# Keep states + DC only (drop territories)
valid_state_fips = {
    "01","02","04","05","06","08","09","10","11","12","13","15","16","17","18","19",
    "20","21","22","23","24","25","26","27","28","29","30","31","32","33","34","35",
    "36","37","38","39","40","41","42","44","45","46","47","48","49","50","51","53",
    "54","55","56"
}
acs_ctrl["state_fips2"] = acs_ctrl["county_fips"].str[:2]
acs_ctrl = acs_ctrl[acs_ctrl["state_fips2"].isin(valid_state_fips)].drop(columns=["state_fips2"])

# Basic hygiene
pct_cols = [c for c in acs_ctrl.columns if c.endswith("_pct") or c.startswith("pct_")]
acs_ctrl[pct_cols] = acs_ctrl[pct_cols].clip(lower=0, upper=100)

ctrl_path = OUT / "acs_controls_2023.csv"
acs_ctrl.to_csv(ctrl_path, index=False)
print("Saved controls:", ctrl_path, "| rows:", len(acs_ctrl), "| cols:", len(acs_ctrl.columns))

Saved controls: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_controls_2023.csv | rows: 3144 | cols: 12


In [ ]:
from pathlib import Path
import pandas as pd

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")

base_df = pd.read_csv(OUT / "acs_rucc_2023_reordered.csv", dtype=str)
base_df["county_fips"] = base_df["county_fips"].str.zfill(5)

ctrl_df = pd.read_csv(OUT / "acs_controls_2023.csv", dtype=str)
ctrl_df["county_fips"] = ctrl_df["county_fips"].str.zfill(5)

# cast numerics for controls
num_cols = [c for c in ctrl_df.columns if c != "county_fips"]
ctrl_df[num_cols] = ctrl_df[num_cols].apply(pd.to_numeric, errors="coerce")

enriched = base_df.merge(ctrl_df, on="county_fips", how="left")

out_path = OUT / "acs_rucc_2023_with_controls.csv"
enriched.to_csv(out_path, index=False)
print("Saved:", out_path, "| rows:", len(enriched), "| cols:", len(enriched.columns))

Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023_with_controls.csv | rows: 3144 | cols: 99


In [ ]:
# Yet another QA check
import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")

df = pd.read_csv(OUT / "acs_rucc_2023_with_controls.csv", dtype=str)
df["county_fips"] = df["county_fips"].str.strip().str.zfill(5)
print("Rows:", len(df), "| Unique counties:", df["county_fips"].nunique())

Rows: 3144 | Unique counties: 3144


In [ ]:
def qc_basic(df, name="df", key="county_fips"):
    print(f"\n=== QC: {name} ===")
    print("Rows:", len(df), "| Cols:", len(df.columns))
    print("Duplicate keys:", df.duplicated(subset=[key]).sum())
    print("Null keys:", df[key].isna().sum())
    bad = df[~df[key].str.match(r"^\d{5}$", na=False)]
    print("Bad FIPS format:", len(bad))

qc_basic(df, "acs_rucc_2023_with_controls")


=== QC: acs_rucc_2023_with_controls ===
Rows: 3144 | Cols: 99
Duplicate keys: 0
Null keys: 0
Bad FIPS format: 0


In [ ]:
# Controls we expect (will only cast those that exist)
controls = [
    "pop_total",
    "median_household_income_usd","per_capita_income_usd",
    "unemployment_rate_pct","pct_hs_or_higher","pct_bachelors_or_higher",
    "pct_households_broadband","pct_households_no_vehicle","pct_households_snap",
    "mean_travel_time_to_work_min","gini_index"
]

for c in controls:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Clip any percent-like columns to [0,100] (just in case)
pct_like = [c for c in df.columns if c.startswith("pct_") or c.endswith("_pct")]
df[pct_like] = df[pct_like].apply(pd.to_numeric, errors="coerce").clip(lower=0, upper=100)

# Quick missingness summary for the controls you actually have
present_controls = [c for c in controls if c in df.columns]
print("\nMissing values in controls:")
print(df[present_controls].isna().sum().sort_values(ascending=False))


Missing values in controls:
pop_total                       0
median_household_income_usd     0
per_capita_income_usd           0
unemployment_rate_pct           0
pct_hs_or_higher                0
pct_bachelors_or_higher         0
pct_households_broadband        0
pct_households_no_vehicle       0
pct_households_snap             0
mean_travel_time_to_work_min    0
gini_index                      0
dtype: int64


In [ ]:
# Basic distribution peeks (only if present)
for c in ["pop_total","median_household_income_usd","unemployment_rate_pct","pct_households_broadband","gini_index"]:
    if c in df.columns:
        print(f"\n{c} describe():")
        print(df[c].describe())

# Metro vs Nonmetro split (households_count) if available
if "households_count" in df.columns and "rucc_metro_flag" in df.columns:
    df["households_count"] = pd.to_numeric(df["households_count"], errors="coerce")
    print("\nHouseholds by Metro/Nonmetro (mean):")
    print(df.groupby("rucc_metro_flag")["households_count"].mean().round(1))


pop_total describe():
count    3.144000e+03
mean     1.057212e+05
std      3.335064e+05
min      4.300000e+01
25%      1.079375e+04
50%      2.581250e+04
75%      6.837925e+04
max      9.848406e+06
Name: pop_total, dtype: float64

median_household_income_usd describe():
count    3.144000e+03
mean    -3.580571e+05
std      1.681344e+07
min     -6.666667e+08
25%      5.497000e+04
50%      6.365400e+04
75%      7.364225e+04
max      1.787070e+05
Name: median_household_income_usd, dtype: float64

unemployment_rate_pct describe():
count    3144.000000
mean        4.741826
std         2.413128
min         0.000000
25%         3.300000
50%         4.500000
75%         5.800000
max        31.100000
Name: unemployment_rate_pct, dtype: float64

pct_households_broadband describe():
count    3144.000000
mean       84.351336
std         6.576653
min        48.100000
25%        81.000000
50%        85.300000
75%        88.900000
max       100.000000
Name: pct_households_broadband, dtype: float64

g

In [ ]:
# Define groups (only keep those that exist)
id_cols = ["county_fips","state_name","state_abbr","county_name"]
rucc_cols = ["rucc_code","rucc_metro_flag","rucc_desc","pop_2020"]
core_cols = ["households_count", "pop_total"]  # add 'land_area_sqmi' later if you add it

control_cols = [
    "median_household_income_usd","per_capita_income_usd","unemployment_rate_pct",
    "pct_hs_or_higher","pct_bachelors_or_higher",
    "pct_households_broadband","pct_households_no_vehicle","pct_households_snap",
    "mean_travel_time_to_work_min","gini_index"
]

race_pct_cols = [
    "pct_one_race_white","pct_one_race_black_or_african_american",
    "pct_one_race_american_indian_and_alaska_native","pct_one_race_asian",
    "pct_one_race_native_hawaiian_and_other_pacific_islander",
    "pct_one_race_some_other_race","pct_households_two_or_more_races",
    "pct_households_hispanic_or_latino_origin_of_any_race",
    "pct_households_white_alone_not_hispanic_or_latino"
]

family_structure_pct_cols = [
    "pct_families_families","pct_families_with_own_children_of_householder_under_18_years",
    "pct_families_with_no_own_children_of_householder_under_18_years",
    "pct_families_married_couple_families","pct_married_couple_families_with_own_children_under_18_years",
    "pct_families_female_householder_no_spouse_present",
    "pct_female_householder_no_spouse_present_with_own_children_under_18_years",
    "pct_families_male_householder_no_spouse_present",
    "pct_male_householder_no_spouse_present_with_own_children_under_18_years",
    "pct_nonfamily_households_nonfamily_households",
    "pct_nonfamily_households_female_householder","pct_female_householder_living_alone",
    "pct_female_householder_not_living_alone","pct_nonfamily_households_male_householder",
    "pct_male_householder_living_alone","pct_male_householder_not_living_alone",
    # income-by-age/earner/family-size % that start with 'pct_household_income...' or 'pct_family_income...'
] + [c for c in df.columns if c.startswith(("pct_household_income_by_", "pct_family_income_by_"))]

median_income_cols = [
    "median_household_income_by_race_and_hispanic_or_latino_origin_of_householder_households_usd",
    "median_one_race_white_usd","median_one_race_black_or_african_american_usd",
    "median_one_race_american_indian_and_alaska_native_usd","median_one_race_asian_usd",
    "median_one_race_native_hawaiian_and_other_pacific_islander_usd",
    "median_one_race_some_other_race_usd","median_households_two_or_more_races_usd",
    "median_households_hispanic_or_latino_origin_of_any_race_usd",
    "median_households_white_alone_not_hispanic_or_latino_usd",
    "median_household_income_by_age_of_householder_15_to_24_years_usd",
    "median_household_income_by_age_of_householder_25_to_44_years_usd",
    "median_household_income_by_age_of_householder_45_to_64_years_usd",
    "median_household_income_by_age_of_householder_65_years_and_over_usd",
    "median_families_families_usd","median_families_with_own_children_of_householder_under_18_years_usd",
    "median_families_with_no_own_children_of_householder_under_18_years_usd",
    "median_families_married_couple_families_usd","median_married_couple_families_with_own_children_under_18_years_usd",
    "median_families_female_householder_no_spouse_present_usd",
    "median_female_householder_no_spouse_present_with_own_children_under_18_years_usd",
    "median_families_male_householder_no_spouse_present_usd",
    "median_male_householder_no_spouse_present_with_own_children_under_18_years_usd",
    "median_family_income_by_family_size_2_person_families_usd",
    "median_family_income_by_family_size_3_person_families_usd",
    "median_family_income_by_family_size_4_person_families_usd",
    "median_family_income_by_family_size_5_person_families_usd",
    "median_family_income_by_family_size_6_person_families_usd",
    "median_family_income_by_family_size_7_or_more_person_families_usd",
    "median_family_income_by_number_of_earners_no_earners_usd",
    "median_family_income_by_number_of_earners_1_earner_usd",
    "median_family_income_by_number_of_earners_2_earners_usd",
    "median_family_income_by_number_of_earners_3_or_more_earners_usd",
    "median_nonfamily_households_nonfamily_households_usd",
    "median_nonfamily_households_female_householder_usd","median_female_householder_living_alone_usd",
    "median_female_householder_not_living_alone_usd","median_nonfamily_households_male_householder_usd",
    "median_male_householder_living_alone_usd","median_male_householder_not_living_alone_usd"
]

# Build final order dynamically (include only columns that exist)
groups = [id_cols, rucc_cols, core_cols, control_cols, race_pct_cols, family_structure_pct_cols, median_income_cols]
ordered = []
seen = set()
for grp in groups:
    for c in grp:
        if c in df.columns and c not in seen:
            ordered.append(c); seen.add(c)

# Append any remaining columns we didn't explicitly order (to avoid data loss)
rest = [c for c in df.columns if c not in seen]
final_cols = ordered + rest

df = df[final_cols].copy()
print("\nFirst 25 columns in new order:\n", df.columns[:25].tolist())


First 25 columns in new order:
 ['county_fips', 'state_name', 'state_abbr', 'county_name', 'rucc_code', 'rucc_metro_flag', 'rucc_desc', 'pop_2020', 'households_count', 'pop_total', 'median_household_income_usd', 'per_capita_income_usd', 'unemployment_rate_pct', 'pct_hs_or_higher', 'pct_bachelors_or_higher', 'pct_households_broadband', 'pct_households_no_vehicle', 'pct_households_snap', 'mean_travel_time_to_work_min', 'gini_index', 'pct_one_race_white', 'pct_one_race_black_or_african_american', 'pct_one_race_american_indian_and_alaska_native', 'pct_one_race_asian', 'pct_one_race_native_hawaiian_and_other_pacific_islander']


In [ ]:
out_path = OUT / "acs_rucc_2023_with_controls_reordered.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path, "| Rows:", len(df), "| Cols:", len(df.columns))

Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023_with_controls_reordered.csv | Rows: 3144 | Cols: 99


MOVING ON TO NaNDA TRACT20!!!

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Folders / files (edit the NaNDA path if yours is different)
OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

nanda_path = "/content/drive/MyDrive/Datasets/nanda_grocery_Tract20_1990-2021_01P.csv"

In [ ]:
# Checking out the file
nanda = pd.read_csv(nanda_path, dtype=str)
print("Rows (all years):", len(nanda))
print("Columns:", nanda.columns.tolist()[:15], "...")
nanda.head(3)

Rows (all years): 2736896
Columns: ['tract_fips20', 'year', 'totpop', 'aland20', 'count_grocery', 'emps_grocery', 'den_grocery', 'aden_grocery', 'count_supermarkets', 'emps_supermarkets', 'den_supermarkets', 'aden_supermarkets', 'count_meatfish', 'emps_meatfish', 'den_meatfish'] ...


,tract_fips20,year,totpop,aland20,count_grocery,emps_grocery,den_grocery,aden_grocery,count_supermarkets,emps_supermarkets,...,den_fruitveg,aden_fruitveg,count_warehousefood,emps_warehousefood,den_warehousefood,aden_warehousefood,count_totalfoodstores,emps_totalfoodstores,den_totalfoodstores,aden_totalfoodstores
0,01001020100,1990,1772.67,3.793571,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,01001020100,1991,1783.301,3.793571,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,01001020100,1992,1793.931,3.793571,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Keep only the most recent year which is 2021
nanda = nanda[nanda["year"] == "2021"].copy()
print("Rows (2021 only):", len(nanda))

Rows (2021 only): 85528


In [ ]:
# Add zero padding to tract ID to 11 characters and then derive the last 5 characters for FIPS
id_col = "tract_fips20" if "tract_fips20" in nanda.columns else "tract_geoid20"
nanda[id_col] = nanda[id_col].str.zfill(11)
nanda["county_fips"] = nanda[id_col].str[:5]
nanda = nanda[nanda["county_fips"].str.match(r"^\d{5}$", na=False)].copy()
print("Unique counties:", nanda["county_fips"].nunique())

Unique counties: 3234


In [ ]:
# Selecting columns to aggregate
count_cols = sorted([c for c in nanda.columns if c.startswith("count_")])
emp_cols   = sorted([c for c in nanda.columns if c.startswith("emps_")])
totals     = [c for c in ["totpop","aland20"] if c in nanda.columns]
print("Counts:", count_cols)
print("Emps:", emp_cols)
print("Totals:", totals)

Counts: ['count_fruitveg', 'count_grocery', 'count_meatfish', 'count_supermarkets', 'count_totalfoodstores', 'count_warehousefood']
Emps: ['emps_fruitveg', 'emps_grocery', 'emps_meatfish', 'emps_supermarkets', 'emps_totalfoodstores', 'emps_warehousefood']
Totals: ['totpop', 'aland20']


In [ ]:
# Convert count and emp columns to numbers and fill empties with 0
for c in count_cols + emp_cols + totals:
    nanda[c] = pd.to_numeric(nanda[c], errors="coerce").fillna(0)

In [ ]:
# Group by county FIPS and sum the selected columns, then rename them
agg_cols = count_cols + emp_cols + totals
nanda_cty = (
    nanda[["county_fips"] + agg_cols]
    .groupby("county_fips", as_index=False)
    .sum()
).rename(columns={c: f"{c}_2021" for c in agg_cols})

print("County rows:", len(nanda_cty))
nanda_cty.head(3)

County rows: 3234


,county_fips,count_fruitveg_2021,count_grocery_2021,count_meatfish_2021,count_supermarkets_2021,count_totalfoodstores_2021,count_warehousefood_2021,emps_fruitveg_2021,emps_grocery_2021,emps_meatfish_2021,emps_supermarkets_2021,emps_totalfoodstores_2021,emps_warehousefood_2021,totpop_2021,aland20_2021
0,01001,0,9,0,1,10,0,0.0,34.0,0.0,3.0,37.0,0.0,58805.0,594.456040
1,01003,3,44,17,24,89,1,8.0,389.0,50.0,1262.5,1759.5,50.0,231767.0,1589.835910
2,01005,1,8,1,3,13,0,1.0,109.0,2.0,93.0,205.0,0.0,25223.0,885.007951


In [ ]:
# Adding metric per 10k people for population standardization
if "totpop_2021" in nanda_cty.columns:
    denom = pd.to_numeric(nanda_cty["totpop_2021"], errors="coerce").replace(0, np.nan)
    for c in count_cols:
        col = f"{c}_2021"
        if col in nanda_cty.columns:
            nanda_cty[f"{c}_per10k_2021"] = (
                pd.to_numeric(nanda_cty[col], errors="coerce") / denom * 1e4
            ).round(3)

In [ ]:
# Adding metric per sq mile for spatial density
if "aland20_2021" in nanda_cty.columns:
    area_sqmi = pd.to_numeric(nanda_cty["aland20_2021"], errors="coerce") / 2_589_988.110336
    area_sqmi = area_sqmi.replace(0, np.nan)
    nanda_cty["area_sqmi_2021"] = area_sqmi.round(6)
    for c in count_cols:
        col = f"{c}_2021"
        if col in nanda_cty.columns:
            nanda_cty[f"{c}_per_sqmi_2021"] = (
                pd.to_numeric(nanda_cty[col], errors="coerce") / area_sqmi
            ).round(6)

In [ ]:
# Checking for duplicates and quick stats
print("Duplicate county_fips:", nanda_cty.duplicated("county_fips").sum())
probe = "count_grocery_2021" if "count_grocery_2021" in nanda_cty.columns else None
if probe:
    print(nanda_cty[probe].describe())

Duplicate county_fips: 0
count    3234.000000
mean       23.491342
std        91.870662
min         0.000000
25%         2.000000
50%         6.000000
75%        14.000000
max      2391.000000
Name: count_grocery_2021, dtype: float64


In [ ]:
# Keeping only states for later merge
valid_state_fips = {
    "01","02","04","05","06","08","09","10","11","12","13","15","16","17","18","19",
    "20","21","22","23","24","25","26","27","28","29","30","31","32","33","34","35",
    "36","37","38","39","40","41","42","44","45","46","47","48","49","50","51","53",
    "54","55","56"
}
nanda_cty["state_fips2"] = nanda_cty["county_fips"].str[:2]
nanda_cty = nanda_cty[nanda_cty["state_fips2"].isin(valid_state_fips)].drop(columns=["state_fips2"])
print("Rows after dropping territories:", len(nanda_cty))

Rows after dropping territories: 3143


In [ ]:
# Reordering columns
id_cols = ["county_fips"]
total_cols = [c for c in ["totpop_2021","area_sqmi_2021","aland20_2021"] if c in nanda_cty.columns]
count_sum_cols = [c for c in nanda_cty.columns if c.startswith("count_") and c.endswith("_2021")]
emp_sum_cols   = [c for c in nanda_cty.columns if c.startswith("emps_") and c.endswith("_2021")]
percap_cols    = [c for c in nanda_cty.columns if c.endswith("_per10k_2021")]
perarea_cols   = [c for c in nanda_cty.columns if c.endswith("_per_sqmi_2021")]

ordered = id_cols + total_cols + count_sum_cols + emp_sum_cols + percap_cols + perarea_cols
rest = [c for c in nanda_cty.columns if c not in ordered]
nanda_cty = nanda_cty[ordered + rest].copy()
print(nanda_cty.columns[:20].tolist())

['county_fips', 'totpop_2021', 'area_sqmi_2021', 'aland20_2021', 'count_fruitveg_2021', 'count_grocery_2021', 'count_meatfish_2021', 'count_supermarkets_2021', 'count_totalfoodstores_2021', 'count_warehousefood_2021', 'count_fruitveg_per10k_2021', 'count_grocery_per10k_2021', 'count_meatfish_per10k_2021', 'count_supermarkets_per10k_2021', 'count_totalfoodstores_per10k_2021', 'count_warehousefood_per10k_2021', 'count_fruitveg_per_sqmi_2021', 'count_grocery_per_sqmi_2021', 'count_meatfish_per_sqmi_2021', 'count_supermarkets_per_sqmi_2021']


In [ ]:
# Saving clean nanda
out_nanda = OUT / "nanda_county_2021_clean.csv"
nanda_cty.to_csv(out_nanda, index=False)
print("Saved:", out_nanda)

Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/nanda_county_2021_clean.csv


Now I will prepare for a QC check on the new nanda file and then cross check with the rucc_acs file prior to merging.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

nanda_path = OUT / "nanda_county_2021_clean.csv"
base_candidates = [
    OUT / "acs_rucc_2023_with_controls_reordered.csv",

]
base_path = next(p for p in base_candidates if p.exists())
print("Base file:", base_path)

Base file: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023_with_controls_reordered.csv


In [ ]:
# Checking for duplcated and format
nanda_cty = pd.read_csv(nanda_path, dtype=str)
nanda_cty["county_fips"] = nanda_cty["county_fips"].str.zfill(5)

print("Rows:", len(nanda_cty), "| Unique county_fips:", nanda_cty["county_fips"].nunique())
print("Duplicate county_fips:", nanda_cty.duplicated("county_fips").sum())
bad_fips = nanda_cty[~nanda_cty["county_fips"].str.match(r"^\d{5}$", na=False)]
print("Bad FIPS format rows:", len(bad_fips))

Rows: 3143 | Unique county_fips: 3143
Duplicate county_fips: 0
Bad FIPS format rows: 0


In [ ]:
# Auto detect columns
count_cols = [c for c in nanda_cty.columns if c.startswith("count_") and c.endswith("_2021")]
emp_cols   = [c for c in nanda_cty.columns if c.startswith("emps_")  and c.endswith("_2021")]
tot_cols   = [c for c in ["totpop_2021","aland20_2021","area_sqmi_2021"] if c in nanda_cty.columns]
percap_cols = [c for c in nanda_cty.columns if c.endswith("_per10k_2021")]
perarea_cols= [c for c in nanda_cty.columns if c.endswith("_per_sqmi_2021")]

# Cast numerics for checks
for c in count_cols + emp_cols + tot_cols + percap_cols + perarea_cols:
    nanda_cty[c] = pd.to_numeric(nanda_cty[c], errors="coerce")

print("Count cols:", count_cols[:8], "..." if len(count_cols)>8 else "")
print("Emps cols :", emp_cols[:8], "..." if len(emp_cols)>8 else "")
print("Totals    :", tot_cols)

Count cols: ['count_fruitveg_2021', 'count_grocery_2021', 'count_meatfish_2021', 'count_supermarkets_2021', 'count_totalfoodstores_2021', 'count_warehousefood_2021', 'count_fruitveg_per10k_2021', 'count_grocery_per10k_2021'] ...
Emps cols : ['emps_fruitveg_2021', 'emps_grocery_2021', 'emps_meatfish_2021', 'emps_supermarkets_2021', 'emps_totalfoodstores_2021', 'emps_warehousefood_2021'] 
Totals    : ['totpop_2021', 'aland20_2021', 'area_sqmi_2021']


In [ ]:
# Checking that there are no negative or missing values
def neg_count(df, cols):
    return {c:int((df[c] < 0).sum()) for c in cols if c in df}

neg_counts = neg_count(nanda_cty, count_cols + emp_cols + tot_cols)
neg_counts = {k:v for k,v in neg_counts.items() if v>0}
print("Negative values (should be 0):", neg_counts)

miss_counts = nanda_cty[count_cols + emp_cols + tot_cols].isna().sum().sort_values(ascending=False)
print("\nMissing values (top 10):")
print(miss_counts.head(10))


Negative values (should be 0): {}

Missing values (top 10):
count_fruitveg_2021               0
count_grocery_2021                0
count_meatfish_2021               0
count_supermarkets_2021           0
count_totalfoodstores_2021        0
count_warehousefood_2021          0
count_fruitveg_per10k_2021        0
count_grocery_per10k_2021         0
count_meatfish_per10k_2021        0
count_supermarkets_per10k_2021    0
dtype: int64


In [ ]:
# Checking distribution snapshot
probe = "count_grocery_2021" if "count_grocery_2021" in nanda_cty.columns else (count_cols[0] if count_cols else None)
if probe:
    print(f"\n{probe} describe():")
    print(nanda_cty[probe].describe())

    print("\nTop 5 counties by", probe)
    display(nanda_cty[["county_fips", probe]].sort_values(probe, ascending=False).head(5))


count_grocery_2021 describe():
count    3143.000000
mean       24.154629
std        93.107384
min         0.000000
25%         3.000000
50%         6.000000
75%        15.000000
max      2391.000000
Name: count_grocery_2021, dtype: float64

Top 5 counties by count_grocery_2021


,county_fips,count_grocery_2021
205,06037,2391
1852,36047,2079
1869,36081,1368
611,17031,1293
2624,48201,1227


In [ ]:
# Ensuring terrioties were dropped
nanda_cty["state_fips2"] = nanda_cty["county_fips"].str[:2]
territories = {"60","66","69","72","78"}  # AS, GU, MP, PR, VI
terr_rows = nanda_cty[nanda_cty["state_fips2"].isin(territories)]
print("Territorial rows (should be 0 if dropped):", len(terr_rows))
nanda_cty = nanda_cty.drop(columns=["state_fips2"])

Territorial rows (should be 0 if dropped): 0


In [ ]:
# Cross checking with base dataset
base = pd.read_csv(base_path, dtype=str)
base["county_fips"] = base["county_fips"].str.zfill(5)

# Ensure unique keys
assert base.duplicated("county_fips").sum()==0, "Base has duplicate county_fips"
assert nanda_cty.duplicated("county_fips").sum()==0, "NaNDA has duplicate county_fips"

only_in_base  = sorted(set(base["county_fips"]) - set(nanda_cty["county_fips"]))
only_in_nanda = sorted(set(nanda_cty["county_fips"]) - set(base["county_fips"]))

print("Only in BASE (not in NaNDA):", len(only_in_base))
print("Only in NaNDA (not in BASE):", len(only_in_nanda))

# Save audits
pd.DataFrame({"county_fips": only_in_base}).to_csv(OUT / "audit_only_in_base.csv", index=False)
pd.DataFrame({"county_fips": only_in_nanda}).to_csv(OUT / "audit_only_in_nanda.csv", index=False)

# Show examples
if only_in_base:
    display(base[base["county_fips"].isin(only_in_base)][["county_fips","state_name","county_name"]].head(10))
if only_in_nanda:
    display(nanda_cty[nanda_cty["county_fips"].isin(only_in_nanda)][["county_fips"] + count_cols[:2]].head(10))

Only in BASE (not in NaNDA): 9
Only in NaNDA (not in BASE): 8


,county_fips,state_name,county_name
309,09110,Connecticut,Capitol Planning Region
310,09120,Connecticut,Greater Bridgeport Planning Region
311,09130,Connecticut,Lower Connecticut River Valley Planning Region
312,09140,Connecticut,Naugatuck Valley Planning Region
313,09150,Connecticut,Northeastern Connecticut Planning Region
314,09160,Connecticut,Northwest Hills Planning Region
315,09170,Connecticut,South Central Connecticut Planning Region
316,09180,Connecticut,Southeastern Connecticut Planning Region
317,09190,Connecticut,Western Connecticut Planning Region


,county_fips,count_fruitveg_2021,count_grocery_2021
309,09001,15,355
310,09003,15,342
311,09005,4,56
312,09007,1,38
313,09009,12,335
314,09011,4,71
315,09013,2,26
316,09015,1,29


The 9 rows only in the BASE are CT’s new Planning Regions with FIPS like 09110–09190. Since 2023, ACS reports planning regions instead of the old 8 counties. The 8 rows only in NaNDA are CT’s old counties (09001–09015). Need to figure out how to map this, if possible, with CT specifi crosswalk.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

nanda_raw_path = "/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/nanda_grocery_Tract20_1990-2021_01P.csv"
xwalk_path     = "/content/drive/MyDrive/GRADTDA5621_Project_I/Data_Sources/2022tractcrosswalk.csv"
base_path      = OUT / "acs_rucc_2023_with_controls_reordered.csv"

In [ ]:
# Selecting only 2021 from the raw nanda file
nanda_21 = pd.read_csv(nanda_raw_path, dtype=str)
nanda_21 = nanda_21[nanda_21["year"]=="2021"].copy()
print("NaNDA 2021 rows:", len(nanda_21))

NaNDA 2021 rows: 85528


In [ ]:
# Zero padding nanda fips to ensure 5 digits
id_col = "tract_fips20" if "tract_fips20" in nanda_21.columns else "tract_geoid20"
nanda_21[id_col] = nanda_21[id_col].str.zfill(11)
nanda_21["county_fips"] = nanda_21[id_col].str[:5]

# Split CT vs non-CT (CT state FIPS = "09")
n_ct  = nanda_21[nanda_21["county_fips"].str[:2]=="09"].copy()
n_non = nanda_21[nanda_21["county_fips"].str[:2]!="09"].copy()

print("CT tracts:", len(n_ct), "| Other states tracts:", len(n_non))

In [ ]:
# Load crosswalk and inspect the headers
xwalk_raw = pd.read_csv(xwalk_path, dtype=str)
xwalk = xwalk_raw.rename(columns=lambda c: c.strip().lower())
print("Crosswalk columns:", list(xwalk.columns))
xwalk.head(3)

Crosswalk columns: ['tract_fips_2020', 'tract_fips_2022', 'tract_name', 'town_name', 'town_fips_2020', 'town_fips_2022', 'county_name', 'county_fips_2020', 'ce_name_2022', 'ce_fips_2022', 'puma2020code', 'puma2020name', 'school_district_code', 'school_district_name', 'zip5_zcta2020']


,tract_fips_2020,tract_fips_2022,tract_name,town_name,town_fips_2020,town_fips_2022,county_name,county_fips_2020,ce_name_2022,ce_fips_2022,puma2020code,puma2020name,school_district_code,school_district_name,zip5_zcta2020
0,09009350400,09140350400,3504,Waterbury,0900980070,0914080070,New Haven,09009,Naugatuck Valley Planning Region,09140,20601,Waterbury Town,151,Waterbury,06704
1,09009350500,09140350500,3505,Waterbury,0900980070,0914080070,New Haven,09009,Naugatuck Valley Planning Region,09140,20601,Waterbury Town,151,Waterbury,06706
2,09009352701,09140352701,3527.01,Waterbury,0900980070,0914080070,New Haven,09009,Naugatuck Valley Planning Region,09140,20601,Waterbury Town,151,Waterbury,06705


In [ ]:
# Lock the exact columns and make formats consistent
tract_col = "tract_fips_2020"
pr_col    = "ce_fips_2022"

xwalk[tract_col] = xwalk[tract_col].astype(str).str.replace(r"\D","", regex=True).str.zfill(11)
xwalk[pr_col]    = xwalk[pr_col].astype(str).str.replace(r"\D","", regex=True).str.zfill(5)

In [ ]:
# Replacing CTs old county FIPS with planning region FIPS
n_ct = n_ct.merge(xwalk[[tract_col, pr_col]], left_on=id_col, right_on=tract_col, how="left")
n_ct["county_fips"] = n_ct[pr_col].fillna(n_ct["county_fips"])

# Creating a helper column
n_ct = n_ct.drop(columns=[tract_col, pr_col], errors="ignore")
print("CT tracts after PR attach:", len(n_ct))

CT tracts after PR attach: 883


In [ ]:
# Remerging to check for CT tracts missing PR FIPS
qa = n_ct.merge(
    xwalk[[tract_col, pr_col]],
    left_on=id_col, right_on=tract_col,
    how="left", suffixes=("","_xwalk")
)

missing = qa[pr_col].isna().sum()
print("CT tracts missing planning-region FIPS:", missing)

# Peek a few if any are missing
if missing:
    display(qa[qa[pr_col].isna()][[id_col, "county_fips"]].head(10))

CT tracts missing planning-region FIPS: 4


,tract_fips20,county_fips
226,09001990000,09001
554,09007990100,09007
753,09009990000,09009
820,09011990100,09011


In [ ]:
# Recreate an unmatched set
unmatched = n_ct.merge(
    xwalk[[tract_col, pr_col]],
    left_on=id_col, right_on=tract_col,
    how="left", suffixes=("","_xwalk")
)
unmatched = unmatched[unmatched[pr_col].isna()].copy()

print("Unmatched CT tracts:", len(unmatched))
display(unmatched[[id_col, "county_fips"]])

# Selecting columns tha contributw to county counts
sum_cols = [c for c in nanda_21.columns if c.startswith(("count_","emps_","totpop","aland20"))]

# Selecting only those that sum to 0 - if match above, then will remove
row_totals = unmatched[sum_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
print("Rows with ALL-ZERO additive values:", int((row_totals==0).sum()), "of", len(unmatched))

Unmatched CT tracts: 4


,tract_fips20,county_fips
226,09001990000,09001
554,09007990100,09007
753,09009990000,09009
820,09011990100,09011


Rows with ALL-ZERO additive values: 4 of 4


In [ ]:
# Removing unmatched 0 CT tracts - likely water according to The Google
if (row_totals==0).all():
    # Drop these from CT before aggregation
    keep_mask = ~n_ct[id_col].isin(unmatched[id_col])
    n_ct = n_ct[keep_mask].copy()
    print("Dropped all-zero unmatched CT tracts:", len(unmatched))
else:
    print("Some unmatched tracts have non-zero values — see Option B below.")

Dropped all-zero unmatched CT tracts: 4


In [ ]:
# Cast numerics and aggregate CT + non-CT
for c in sum_cols:
    n_ct[c]  = pd.to_numeric(n_ct[c], errors="coerce").fillna(0)
    n_non[c] = pd.to_numeric(n_non[c], errors="coerce").fillna(0)

n_ct_agg  = n_ct.groupby("county_fips", as_index=False)[sum_cols].sum()
n_non_agg = n_non.groupby("county_fips", as_index=False)[sum_cols].sum()

nanda_cty_pr = pd.concat([n_non_agg, n_ct_agg], ignore_index=True)
nanda_cty_pr = nanda_cty_pr.rename(columns={c: f"{c}_2021" for c in sum_cols})

In [ ]:
# Cross QC base vs nanda
base = pd.read_csv(OUT / "acs_rucc_2023_with_controls_reordered.csv", dtype=str)
base["county_fips"] = base["county_fips"].str.zfill(5)

only_in_base  = sorted(set(base["county_fips"]) - set(nanda_cty_pr["county_fips"]))
only_in_nanda = sorted(set(nanda_cty_pr["county_fips"]) - set(base["county_fips"]))
print("Only in BASE (not in NaNDA):", len(only_in_base))
print("Only in NaNDA (not in BASE):", len(only_in_nanda))

out_path = OUT / "nanda_county_2021_clean_prfixed.csv"
nanda_cty_pr.to_csv(out_path, index=False)
print("Saved:", out_path)

Only in BASE (not in NaNDA): 0
Only in NaNDA (not in BASE): 91
Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/nanda_county_2021_clean_prfixed.csv


In [ ]:
import pandas as pd
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")

nanda_cty = pd.read_csv(OUT / "nanda_county_2021_clean_prfixed.csv", dtype=str)
nanda_cty["county_fips"] = nanda_cty["county_fips"].str.zfill(5)
nanda_cty["state_fips2"] = nanda_cty["county_fips"].str[:2]

# Checking what those 91 only in nanda are
print("Counts by state_fips2 before filter:")
print(nanda_cty["state_fips2"].value_counts().sort_index())

# Remove territories (AS, GU, MP, PR, VI)
territories = {"60","66","69","72","78"}

nanda_std = nanda_cty[~nanda_cty["state_fips2"].isin(territories)].drop(columns=["state_fips2"])
print("Rows before:", len(nanda_cty), "| after (states+DC only):", len(nanda_std))

nanda_std.to_csv(OUT / "nanda_county_2021_clean_statesdc.csv", index=False)
print("Saved:", OUT / "nanda_county_2021_clean_statesdc.csv")

Counts by state_fips2 before filter:
state_fips2
01     67
02     30
04     15
05     75
06     58
08     64
09      9
10      3
11      1
12     67
13    159
15      5
16     44
17    102
18     92
19     99
20    105
21    120
22     64
23     16
24     24
25     14
26     83
27     87
28     82
29    115
30     56
31     93
32     17
33     10
34     21
35     33
36     62
37    100
38     53
39     88
40     77
41     36
42     67
44      5
45     46
46     66
47     95
48    254
49     29
50     14
51    133
53     39
54     55
55     72
56     23
60      5
66      1
69      4
72     78
78      3
Name: count, dtype: int64
Rows before: 3235 | after (states+DC only): 3144
Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/nanda_county_2021_clean_statesdc.csv


In [ ]:
# Cross QC again to ensure 0/0
base = pd.read_csv(OUT / "acs_rucc_2023_with_controls_reordered.csv", dtype=str)
base["county_fips"] = base["county_fips"].str.zfill(5)

only_in_base  = sorted(set(base["county_fips"]) - set(nanda_std["county_fips"]))
only_in_nanda = sorted(set(nanda_std["county_fips"]) - set(base["county_fips"]))

print("Only in BASE (not in NaNDA):", len(only_in_base))
print("Only in NaNDA (not in BASE):", len(only_in_nanda))

Only in BASE (not in NaNDA): 0
Only in NaNDA (not in BASE): 0


In [ ]:
# Merging nanda to base
merged = base.merge(nanda_std, on="county_fips", how="left")
print("Merged rows:", len(merged), "| Unique counties:", merged["county_fips"].nunique())

out_merged = OUT / "acs_rucc_controls_nanda_2021.csv"
merged.to_csv(out_merged, index=False)
print("Saved:", out_merged)

Merged rows: 3144 | Unique counties: 3144
Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021.csv


In [ ]:
# Readding metrics
import numpy as np
merged = pd.read_csv(out_merged, dtype=str)

# Per 10k using ACS pop_total
if "pop_total" in merged.columns:
    merged["pop_total"] = pd.to_numeric(merged["pop_total"], errors="coerce").replace(0, np.nan)
    for c in [col for col in merged.columns if col.startswith("count_") and col.endswith("_2021")]:
        merged[f"{c.replace('_2021','')}_per10k_2021"] = (
            pd.to_numeric(merged[c], errors="coerce") / merged["pop_total"] * 1e4
        ).round(3)

# Per sq mi using ACS land_area_sqmi
if "land_area_sqmi" in merged.columns:
    area = pd.to_numeric(merged["land_area_sqmi"], errors="coerce").replace(0, np.nan)
    for c in [col for col in merged.columns if col.startswith("count_") and col.endswith("_2021")]:
        merged[f"{c.replace('_2021','')}_per_sqmi_2021"] = (
            pd.to_numeric(merged[c], errors="coerce") / area
        ).round(6)

merged.to_csv(out_merged, index=False)
print("Updated with derived metrics:", out_merged)

Updated with derived metrics: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021.csv


The final merge has been completed. However there were issues with adding per10k and sqmi metrics so I backtracked and readded them.

In [ ]:
from pathlib import Path
import glob

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")

# likely pre-QC file (post-merge, before flags/derived):
candidates = [
    OUT / "acs_rucc_controls_nanda_2021.csv"
]
print("Exists:")
for p in candidates:
    print("  ", p, "->", p.exists())

print("\nAll CSVs I can see:")
print("\n".join(sorted(glob.glob(str(OUT / "*.csv")))))

Exists:
   /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021.csv -> True

All CSVs I can see:
/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_controls_2023.csv
/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_county_2023_clean.csv
/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023.csv
/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023_clean.csv
/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023_reordered.csv
/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023_with_controls.csv
/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_2023_with_controls_reordered.csv
/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021.csv
/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/audit_only_in_acs.csv
/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/audit_only_in_base.csv
/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/audit_only_in_nanda.c

In [ ]:
import pandas as pd

base_path = OUT / "acs_rucc_controls_nanda_2021.csv"
df = pd.read_csv(base_path, dtype=str)
df["county_fips"] = df["county_fips"].str.zfill(5)

print("Backed up to:", base_path, "| rows:", len(df))

Backed up to: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021.csv | rows: 3144


In [ ]:
print(list(df.columns))
print("Total columns:", len(df.columns))

['county_fips', 'state_name', 'state_abbr', 'county_name', 'rucc_code', 'rucc_metro_flag', 'rucc_desc', 'pop_2020', 'households_count', 'pop_total', 'median_household_income_usd', 'per_capita_income_usd', 'unemployment_rate_pct', 'pct_hs_or_higher', 'pct_bachelors_or_higher', 'pct_households_broadband', 'pct_households_no_vehicle', 'pct_households_snap', 'mean_travel_time_to_work_min', 'gini_index', 'pct_one_race_white', 'pct_one_race_black_or_african_american', 'pct_one_race_american_indian_and_alaska_native', 'pct_one_race_asian', 'pct_one_race_native_hawaiian_and_other_pacific_islander', 'pct_one_race_some_other_race', 'pct_households_two_or_more_races', 'pct_households_hispanic_or_latino_origin_of_any_race', 'pct_households_white_alone_not_hispanic_or_latino', 'pct_families_families', 'pct_families_with_own_children_of_householder_under_18_years', 'pct_families_with_no_own_children_of_householder_under_18_years', 'pct_families_married_couple_families', 'pct_married_couple_families_

In [ ]:
# Checking if metrics were added
per10k_cols = [c for c in df.columns if c.endswith("_per10k_2021")]
persqmi_cols = [c for c in df.columns if c.endswith("_per_sqmi_2021")]

print("per10k cols:", len(per10k_cols))
print("per_sqmi cols:", len(persqmi_cols))

per10k cols: 6
per_sqmi cols: 0


In [ ]:
area = None
if "land_area_sqmi" in df.columns:
    area = pd.to_numeric(df["land_area_sqmi"], errors="coerce").replace(0, np.nan)
elif "aland20_2021" in df.columns:
    area = (pd.to_numeric(df["aland20_2021"], errors="coerce") / 2_589_988.110336).replace(0, np.nan)
else:
    print("No area denominator found (need land_area_sqmi or aland20_2021).")

In [ ]:
count21_cols = [c for c in df.columns if c.startswith("count_") and c.endswith("_2021")]

if area is not None and count21_cols:
    for c in count21_cols:
        base = c.replace("_2021","")
        df[f"{base}_per_sqmi_2021"] = (
            pd.to_numeric(df[c], errors="coerce") / area
        ).replace([np.inf, -np.inf], np.nan).round(6)
else:
    print("Skipped: missing area or count columns.")

In [ ]:
added = [c for c in df.columns if c.endswith("_per_sqmi_2021")]
print("per_sqmi columns added:", len(added), "->", added[:6], "..." if len(added)>6 else "")

out_path = OUT / "acs_rucc_controls_nanda_2021_eda.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)

per_sqmi columns added: 12 -> ['count_grocery_per_sqmi_2021', 'count_supermarkets_per_sqmi_2021', 'count_meatfish_per_sqmi_2021', 'count_fruitveg_per_sqmi_2021', 'count_warehousefood_per_sqmi_2021', 'count_totalfoodstores_per_sqmi_2021'] ...
Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_eda.csv


In [ ]:
print(list(df.columns))
print("Total columns:", len(df.columns))

['county_fips', 'state_name', 'state_abbr', 'county_name', 'rucc_code', 'rucc_metro_flag', 'rucc_desc', 'pop_2020', 'households_count', 'pop_total', 'median_household_income_usd', 'per_capita_income_usd', 'unemployment_rate_pct', 'pct_hs_or_higher', 'pct_bachelors_or_higher', 'pct_households_broadband', 'pct_households_no_vehicle', 'pct_households_snap', 'mean_travel_time_to_work_min', 'gini_index', 'pct_one_race_white', 'pct_one_race_black_or_african_american', 'pct_one_race_american_indian_and_alaska_native', 'pct_one_race_asian', 'pct_one_race_native_hawaiian_and_other_pacific_islander', 'pct_one_race_some_other_race', 'pct_households_two_or_more_races', 'pct_households_hispanic_or_latino_origin_of_any_race', 'pct_households_white_alone_not_hispanic_or_latino', 'pct_families_families', 'pct_families_with_own_children_of_householder_under_18_years', 'pct_families_with_no_own_children_of_householder_under_18_years', 'pct_families_married_couple_families', 'pct_married_couple_families_

In [ ]:
drop_cols = df.filter(regex=r'(_per10k_2021|_per_sqmi_2021)$', axis=1).columns.tolist()
print("Will drop:", len(drop_cols), "columns")
print(drop_cols[:10], "..." if len(drop_cols)>10 else "")

Will drop: 18 columns
['count_grocery_per10k_2021', 'count_supermarkets_per10k_2021', 'count_meatfish_per10k_2021', 'count_fruitveg_per10k_2021', 'count_warehousefood_per10k_2021', 'count_totalfoodstores_per10k_2021', 'count_grocery_per_sqmi_2021', 'count_supermarkets_per_sqmi_2021', 'count_meatfish_per_sqmi_2021', 'count_fruitveg_per_sqmi_2021'] ...


In [ ]:
df = df.drop(columns=drop_cols, errors="ignore").copy()
print("Columns after drop:", len(df.columns))

Columns after drop: 113


In [ ]:
df.to_csv(OUT / "acs_rucc_controls_nanda_2021_no_rates.csv", index=False)
print("Saved:", OUT / "acs_rucc_controls_nanda_2021_no_rates.csv")

Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_no_rates.csv


In [ ]:
count21_cols = [c for c in df.columns if c.startswith("count_") and c.endswith("_2021")]
print("count cols:", len(count21_cols))

count cols: 6


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
df = pd.read_csv(OUT / "acs_rucc_controls_nanda_2021_no_rates.csv", dtype=str)
df["county_fips"] = df["county_fips"].str.zfill(5)

In [ ]:
# All NaNDA count columns to convert
count21_cols = [c for c in df.columns if c.startswith("count_") and c.endswith("_2021")]

# Population denominator (prefer ACS)
if "pop_total" in df.columns:
    pop = pd.to_numeric(df["pop_total"], errors="coerce").replace(0, np.nan)
elif "totpop_2021" in df.columns:
    pop = pd.to_numeric(df["totpop_2021"], errors="coerce").replace(0, np.nan)
else:
    pop = None  # per-10k will be skipped

# Area denominator (prefer ACS land area if you have it; else NaNDA aland20_2021 in m²)
if "land_area_sqmi" in df.columns:
    area = pd.to_numeric(df["land_area_sqmi"], errors="coerce").replace(0, np.nan)
elif "aland20_2021" in df.columns:
    area = (pd.to_numeric(df["aland20_2021"], errors="coerce") / 2_589_988.110336).replace(0, np.nan)
else:
    area = None  # per-sqmi will be skipped

In [ ]:
# Create new columns in separate DataFrames
per10k_df = pd.DataFrame()
if pop is not None:
    per10k_df = pd.DataFrame({
        f"{c.replace('_2021','')}_per10k_2021": (pd.to_numeric(df[c], errors="coerce") / pop * 1e4)
        for c in count21_cols
    }).replace([np.inf, -np.inf], np.nan).round(3)

persqmi_df = pd.DataFrame()
if area is not None:
    persqmi_df = pd.DataFrame({
        f"{c.replace('_2021','')}_per_sqmi_2021": (pd.to_numeric(df[c], errors="coerce") / area)
        for c in count21_cols
    }).replace([np.inf, -np.inf], np.nan).round(6)

# Combine both families at once
new_rates = pd.concat([per10k_df, persqmi_df], axis=1)

In [ ]:
# Remove old versions if they exist to avoid duplicates
to_drop = [c for c in new_rates.columns if c in df.columns]
df = df.drop(columns=to_drop, errors="ignore")

# Single concat — avoids fragmentation warnings
df = pd.concat([df, new_rates], axis=1)

In [ ]:
print("per-10k cols:", len([c for c in df.columns if c.endswith("_per10k_2021")]))
print("per-sqmi cols:", len([c for c in df.columns if c.endswith("_per_sqmi_2021")]))

out_path = OUT / "acs_rucc_controls_nanda_2021_rates.csv"
print("Saved:", out_path)

per-10k cols: 6
per-sqmi cols: 6
Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_rates.csv


In [ ]:
# all headers as a Python list
print(list(df.columns))

# count + first/last few
print("Total cols:", len(df.columns))

['county_fips', 'state_name', 'state_abbr', 'county_name', 'rucc_code', 'rucc_metro_flag', 'rucc_desc', 'pop_2020', 'households_count', 'pop_total', 'median_household_income_usd', 'per_capita_income_usd', 'unemployment_rate_pct', 'pct_hs_or_higher', 'pct_bachelors_or_higher', 'pct_households_broadband', 'pct_households_no_vehicle', 'pct_households_snap', 'mean_travel_time_to_work_min', 'gini_index', 'pct_one_race_white', 'pct_one_race_black_or_african_american', 'pct_one_race_american_indian_and_alaska_native', 'pct_one_race_asian', 'pct_one_race_native_hawaiian_and_other_pacific_islander', 'pct_one_race_some_other_race', 'pct_households_two_or_more_races', 'pct_households_hispanic_or_latino_origin_of_any_race', 'pct_households_white_alone_not_hispanic_or_latino', 'pct_families_families', 'pct_families_with_own_children_of_householder_under_18_years', 'pct_families_with_no_own_children_of_householder_under_18_years', 'pct_families_married_couple_families', 'pct_married_couple_families_

In [ ]:
df.to_csv(OUT / "acs_rucc_controls_nanda_2021_rates.csv", index=False)
print("Saved:", OUT / "acs_rucc_controls_nanda_2021_rates.csv")

Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_rates.csv


In [ ]:
import pandas as pd
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
df = pd.read_csv(OUT / "acs_rucc_controls_nanda_2021_rates.csv", dtype=str)

desired = [
    'county_fips','state_name','state_abbr','county_name',
    'rucc_code','rucc_metro_flag','rucc_desc','pop_2020',
    'households_count','pop_total','median_household_income_usd','per_capita_income_usd',
    'unemployment_rate_pct','pct_hs_or_higher','pct_bachelors_or_higher',
    'pct_households_broadband','pct_households_no_vehicle','pct_households_snap',
    'mean_travel_time_to_work_min','gini_index',
    'pct_one_race_white','pct_one_race_black_or_african_american',
    'pct_one_race_american_indian_and_alaska_native','pct_one_race_asian',
    'pct_one_race_native_hawaiian_and_other_pacific_islander','pct_one_race_some_other_race',
    'pct_households_two_or_more_races','pct_households_hispanic_or_latino_origin_of_any_race',
    'pct_households_white_alone_not_hispanic_or_latino',
    'pct_families_families','pct_families_with_own_children_of_householder_under_18_years',
    'pct_families_with_no_own_children_of_householder_under_18_years',
    'pct_families_married_couple_families','pct_married_couple_families_with_own_children_under_18_years',
    'pct_families_female_householder_no_spouse_present',
    'pct_female_householder_no_spouse_present_with_own_children_under_18_years',
    'pct_families_male_householder_no_spouse_present',
    'pct_male_householder_no_spouse_present_with_own_children_under_18_years',
    'pct_nonfamily_households_nonfamily_households','pct_nonfamily_households_female_householder',
    'pct_female_householder_living_alone','pct_female_householder_not_living_alone',
    'pct_nonfamily_households_male_householder','pct_male_householder_living_alone',
    'pct_male_householder_not_living_alone',
    'pct_household_income_by_age_of_householder_15_to_24_years',
    'pct_household_income_by_age_of_householder_25_to_44_years',
    'pct_household_income_by_age_of_householder_45_to_64_years',
    'pct_household_income_by_age_of_householder_65_years_and_over',
    'pct_family_income_by_family_size_2_person_families',
    'pct_family_income_by_family_size_3_person_families',
    'pct_family_income_by_family_size_4_person_families',
    'pct_family_income_by_family_size_5_person_families',
    'pct_family_income_by_family_size_6_person_families',
    'pct_family_income_by_family_size_7_or_more_person_families',
    'pct_family_income_by_number_of_earners_no_earners',
    'pct_family_income_by_number_of_earners_1_earner',
    'pct_family_income_by_number_of_earners_2_earners',
    'pct_family_income_by_number_of_earners_3_or_more_earners',
    'median_household_income_by_race_and_hispanic_or_latino_origin_of_householder_households_usd',
    'median_one_race_white_usd','median_one_race_black_or_african_american_usd',
    'median_one_race_american_indian_and_alaska_native_usd','median_one_race_asian_usd',
    'median_one_race_native_hawaiian_and_other_pacific_islander_usd',
    'median_one_race_some_other_race_usd','median_households_two_or_more_races_usd',
    'median_households_hispanic_or_latino_origin_of_any_race_usd',
    'median_households_white_alone_not_hispanic_or_latino_usd',
    'median_household_income_by_age_of_householder_15_to_24_years_usd',
    'median_household_income_by_age_of_householder_25_to_44_years_usd',
    'median_household_income_by_age_of_householder_45_to_64_years_usd',
    'median_household_income_by_age_of_householder_65_years_and_over_usd',
    'median_families_families_usd','median_families_with_own_children_of_householder_under_18_years_usd',
    'median_families_with_no_own_children_of_householder_under_18_years_usd',
    'median_families_married_couple_families_usd',
    'median_married_couple_families_with_own_children_under_18_years_usd',
    'median_families_female_householder_no_spouse_present_usd',
    'median_families_male_householder_no_spouse_present_usd',
    'median_family_income_by_family_size_2_person_families_usd',
    'median_family_income_by_family_size_3_person_families_usd',
    'median_family_income_by_family_size_4_person_families_usd',
    'median_family_income_by_family_size_5_person_families_usd',
    'median_family_income_by_family_size_6_person_families_usd',
    'median_family_income_by_family_size_7_or_more_person_families_usd',
    'median_family_income_by_number_of_earners_no_earners_usd',
    'median_family_income_by_number_of_earners_1_earner_usd',
    'median_family_income_by_number_of_earners_2_earners_usd',
    'median_family_income_by_number_of_earners_3_or_more_earners_usd',
    'median_nonfamily_households_nonfamily_households_usd',
    'median_nonfamily_households_female_householder_usd',
    'median_female_householder_living_alone_usd',
    'median_female_householder_not_living_alone_usd',
    'median_nonfamily_households_male_householder_usd',
    'median_male_householder_living_alone_usd',
    'median_male_householder_not_living_alone_usd',
    'totpop_2021','aland20_2021',
    'count_grocery_2021','emps_grocery_2021',
    'count_supermarkets_2021','emps_supermarkets_2021',
    'count_meatfish_2021','emps_meatfish_2021',
    'count_fruitveg_2021','emps_fruitveg_2021',
    'count_warehousefood_2021','emps_warehousefood_2021',
    'count_totalfoodstores_2021','emps_totalfoodstores_2021',
    'count_grocery_per10k_2021','count_supermarkets_per10k_2021',
    'count_meatfish_per10k_2021','count_fruitveg_per10k_2021',
    'count_warehousefood_per10k_2021','count_totalfoodstores_per10k_2021',
    'count_grocery_per_sqmi_2021','count_supermarkets_per_sqmi_2021',
    'count_meatfish_per_sqmi_2021','count_fruitveg_per_sqmi_2021',
    'count_warehousefood_per_sqmi_2021','count_totalfoodstores_per_sqmi_2021'
]

present = [c for c in desired if c in df.columns]
missing = [c for c in desired if c not in df.columns]
rest    = [c for c in df.columns if c not in present]

print("Present from desired list:", len(present))
print("Missing from desired list:", len(missing))

# Reorder in one shot
df = df[present + rest].copy()

# Save (new filename so you keep the original too)
out_path = OUT / "acs_rucc_controls_nanda_2021_rates_reordered.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)
print("First 20 columns:", list(df.columns[:20]))

Present from desired list: 123
Missing from desired list: 0
Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_rates_reordered.csv
First 20 columns: ['county_fips', 'state_name', 'state_abbr', 'county_name', 'rucc_code', 'rucc_metro_flag', 'rucc_desc', 'pop_2020', 'households_count', 'pop_total', 'median_household_income_usd', 'per_capita_income_usd', 'unemployment_rate_pct', 'pct_hs_or_higher', 'pct_bachelors_or_higher', 'pct_households_broadband', 'pct_households_no_vehicle', 'pct_households_snap', 'mean_travel_time_to_work_min', 'gini_index']


Now that everything is reordered and metrics are added, I will now check for missing data and fill empty spots as needed.

In [ ]:
rows = len(df)
miss = df.isna().sum().sort_values(ascending=False)
report = (miss.to_frame("missing_rows")
          .assign(missing_pct=lambda x: (x["missing_rows"]/rows*100).round(2))
          .rename_axis("column").reset_index())

# view
print("missing (NaN only):")
display(report.head(50))

# just the columns that have any missing
missing_cols_nan = report.loc[report["missing_rows"]>0, "column"].tolist()
print("Columns with any NaN missing:", len(missing_cols_nan))


missing (NaN only):


,column,missing_rows,missing_pct
0,median_one_race_native_hawaiian_and_other_paci...,2891,91.95
1,median_one_race_asian_usd,1754,55.79
2,median_one_race_american_indian_and_alaska_nat...,1741,55.38
3,median_one_race_black_or_african_american_usd,1296,41.22
4,median_one_race_some_other_race_usd,1218,38.74
5,median_family_income_by_family_size_7_or_more_...,1001,31.84
6,median_households_hispanic_or_latino_origin_of...,647,20.58
7,median_family_income_by_family_size_6_person_f...,634,20.17
8,median_male_householder_no_spouse_present_with...,587,18.67
9,median_female_householder_not_living_alone_usd,568,18.07


Columns with any NaN missing: 40


In [ ]:
import numpy as np
df_blank_as_na = df.replace(r"^\s*$", np.nan, regex=True)

rows2 = len(df_blank_as_na)
miss2 = df_blank_as_na.isna().sum().sort_values(ascending=False)
report2 = (miss2.to_frame("missing_rows")
           .assign(missing_pct=lambda x: (x["missing_rows"]/rows2*100).round(2))
           .rename_axis("column").reset_index())

print("Missing (NaN + blanks):")
display(report2.head(50))

missing_cols_nan_or_blank = report2.loc[report2["missing_rows"]>0, "column"].tolist()
print("Columns with any missing (NaN or blank):", len(missing_cols_nan_or_blank))


Missing (NaN + blanks):


,column,missing_rows,missing_pct
0,median_one_race_native_hawaiian_and_other_paci...,2891,91.95
1,median_one_race_asian_usd,1754,55.79
2,median_one_race_american_indian_and_alaska_nat...,1741,55.38
3,median_one_race_black_or_african_american_usd,1296,41.22
4,median_one_race_some_other_race_usd,1218,38.74
5,median_family_income_by_family_size_7_or_more_...,1001,31.84
6,median_households_hispanic_or_latino_origin_of...,647,20.58
7,median_family_income_by_family_size_6_person_f...,634,20.17
8,median_male_householder_no_spouse_present_with...,587,18.67
9,median_female_householder_not_living_alone_usd,568,18.07


Columns with any missing (NaN or blank): 40


For the missing data points, the following will be applied:

If >30% missing: Keep NA; don’t impute; exclude from models. Optionally keep for descriptive tables with a coverage note.

If 5–30% missing: Prefer models that handle NA (trees/boosting) + add a missingness indicator; or impute with a grouped median (e.g., by state_name × rucc_metro_flag), then run a sensitivity check.

If <5% missing:
Simple state-level median (or state_name × rucc_metro_flag) imputation is acceptable, plus a missingness indicator.

In [ ]:
import numpy as np
import pandas as pd

rows = len(df)
miss = (df.isna().sum()/rows).sort_values(ascending=False)

high_miss = miss[miss >= 0.30].index.tolist()     # drop from models
mid_miss  = miss[(miss >= 0.05) & (miss < 0.30)].index.tolist()
low_miss  = miss[(miss > 0.0) & (miss < 0.05)].index.tolist()

print("High-missing (>=30%):", len(high_miss))
print("Mid-missing (5–30%):", len(mid_miss))
print("Low-missing (<5%):", len(low_miss))

High-missing (>=30%): 6
Mid-missing (5–30%): 10
Low-missing (<5%): 24


In [ ]:
# Add *_is_missing flags (useful features & documentation)
for c in mid_miss + low_miss:
    df[f"{c}__is_missing"] = df[c].isna().astype(int)

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M")
ckpt_csv = OUT / f"acs_rucc_controls_nanda_2021_eda_pre_missing_{stamp}.csv"

df.to_csv(ckpt_csv, index=False)
print("Saved checkpoint:", ckpt_csv)

Saved checkpoint: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_eda_pre_missing_20250927_0048.csv


In [ ]:
# Impute for low missing
group_keys = ["state_name","rucc_metro_flag"]
for c in low_miss:
    df[c] = pd.to_numeric(df[c], errors="coerce")

    # group medians
    g12 = df.groupby(group_keys)[c].transform("median")
    g1  = df.groupby(["state_name"])[c].transform("median")
    g0  = df[c].median()

    # fill in order: group -> state -> national
    df[c] = df[c].fillna(g12).fillna(g1).fillna(g0)

In [ ]:
# Impute for mid missing
group_keys = ["state_name","rucc_metro_flag"]

filled_counts = {}
for c in mid_miss:
    # cast numeric
    df[c] = pd.to_numeric(df[c], errors="coerce")
    before = df[c].isna().sum()

    # group medians
    g12 = df.groupby(group_keys)[c].transform("median") if all(k in df.columns for k in group_keys) else None
    g1  = df.groupby(["state_name"])[c].transform("median") if "state_name" in df.columns else None
    g0  = df[c].median()

    # fill in order: group -> state -> national
    if g12 is not None: df[c] = df[c].fillna(g12)
    if g1  is not None: df[c] = df[c].fillna(g1)
    df[c] = df[c].fillna(g0)

    after = df[c].isna().sum()
    filled_counts[c] = int(before - after)

print("Filled cells (mid-miss):", {k:v for k,v in filled_counts.items() if v})

Filled cells (mid-miss): {'median_households_hispanic_or_latino_origin_of_any_race_usd': 647, 'median_family_income_by_family_size_6_person_families_usd': 634, 'median_male_householder_no_spouse_present_with_own_children_under_18_years_usd': 587, 'median_female_householder_not_living_alone_usd': 568, 'median_household_income_by_age_of_householder_15_to_24_years_usd': 498, 'median_male_householder_not_living_alone_usd': 455, 'median_households_two_or_more_races_usd': 394, 'median_female_householder_no_spouse_present_with_own_children_under_18_years_usd': 319, 'median_families_male_householder_no_spouse_present_usd': 249, 'median_family_income_by_family_size_5_person_families_usd': 221}


In [ ]:
# Verify remaining missing for BOTH mid_miss and low_miss
try:
    mid_miss, low_miss
except NameError:
    rows = len(df)
    miss_pct = df.isna().sum().div(rows)
    mid_miss = miss_pct[(miss_pct >= 0.05) & (miss_pct < 0.30)].index.tolist()
    low_miss = miss_pct[(miss_pct > 0.0) & (miss_pct < 0.05)].index.tolist()

def verify_missing(df, cols, label):
    if cols:
        rem = df[cols].isna().sum().sort_values(ascending=False)
        print(f"\nRemaining NA in {label} columns (top 20):")
        print(rem[rem > 0].head(20))
        print(f"Cols still with any NA in {label}: {(rem > 0).sum()} / {len(cols)}")
    else:
        print(f"\nNo {label} columns detected.")

verify_missing(df, mid_miss, "mid_miss (5–30%)")
verify_missing(df, low_miss, "low_miss (<5%)")


Remaining NA in mid_miss (5–30%) columns (top 20):
Series([], dtype: int64)
Cols still with any NA in mid_miss (5–30%): 0 / 10

Remaining NA in low_miss (<5%) columns (top 20):
Series([], dtype: int64)
Cols still with any NA in low_miss (<5%): 0 / 24


In [ ]:
# Saved post-impute copy
from pathlib import Path
OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
outp = OUT / "acs_rucc_controls_nanda_2021_imputed.csv"
df.to_csv(outp, index=False)
print("Saved:", outp)

Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_imputed.csv


In [ ]:
# Creating a copy where high miss columns are filtered out
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M")

# Ensuring high_miss exists
try:
    high_miss
except NameError:
    rows = len(df)
    miss_pct = df.isna().sum().div(rows)
    high_miss = miss_pct[miss_pct >= 0.30].index.tolist()

print("High-missing cols (>=30%):", len(high_miss))

# Filtering the high-miss columns (and any matching __is_missing flags if present)
to_drop = set(high_miss) | {
    f"{c}__is_missing" for c in high_miss if f"{c}__is_missing" in df.columns
}

model_df = df.drop(columns=list(to_drop), errors="ignore").copy()

# Saving model-ready copy + drop log
model_path = OUT / f"acs_rucc_controls_nanda_2021_imputed_highfilter_{stamp}.csv"
model_df.to_csv(model_path, index=False)

log_path = OUT / f"dropped_high_miss_columns_{stamp}.csv"
pd.Series(sorted(to_drop), name="dropped_column").to_csv(log_path, index=False)

print("Saved model-ready:", model_path)
print("Saved drop log:", log_path)

# Quick verify
still_there = [c for c in high_miss if c in model_df.columns]
print("High-miss columns still present (should be 0):", len(still_there))
print("Shape before/after:", df.shape, "->", model_df.shape)


High-missing cols (>=30%): 6
Saved model-ready: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_imputed_highfilter_20250927_0129.csv
Saved drop log: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/dropped_high_miss_columns_20250927_0129.csv
High-miss columns still present (should be 0): 0
Shape before/after: (3144, 159) -> (3144, 153)


At this point, the merged data set has been analyzed for missing data points. Anything below 30% has been imputed using the median value and anything over 30% has been filtered out. Both files exist, but the filtered file is more appropriate for EDA.

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M")
out_path = OUT / f"acs_rucc_controls_nanda_2021_imputed_highfilter.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_imputed_highfilter.csv


Will peform some EDA prep.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
file_path = OUT / "acs_rucc_controls_nanda_2021_imputed_highfilter.csv"

df = pd.read_csv(file_path, dtype=str)
df["county_fips"] = df["county_fips"].str.zfill(5)
print("Rows:", len(df))

Rows: 3144


In [ ]:
assert df["county_fips"].notna().all() and df["county_fips"].str.match(r"^\d{5}$").all(), "Bad FIPS"
print("Duplicate county_fips (should be 0):", df.duplicated("county_fips").sum())

Duplicate county_fips (should be 0): 0


In [ ]:
num_like = ("count_","emps_","median_","pct_","pop_","totpop_","gini_index","mean_travel_time_to_work_min","aland20_2021","land_area_sqmi")
num_cols = [c for c in df.columns if c.startswith(num_like) or c.endswith(("_per10k_2021","_per_sqmi_2021"))]
for c in num_cols: df[c] = pd.to_numeric(df[c], errors="coerce")

if "rucc_metro_flag" in df.columns: df["rucc_metro_flag"] = df["rucc_metro_flag"].astype("category")
if "state_abbr" in df.columns: df["state_abbr"] = df["state_abbr"].astype("category")
if "rucc_code" in df.columns: df["rucc_code"] = pd.to_numeric(df["rucc_code"], errors="coerce").astype("Int64")

In [ ]:
# pop density (if area available)
if ("pop_total" in df.columns) and (("land_area_sqmi" in df.columns) or ("aland20_2021" in df.columns)):
    area_sqmi = (pd.to_numeric(df["land_area_sqmi"], errors="coerce")
                 if "land_area_sqmi" in df.columns
                 else pd.to_numeric(df["aland20_2021"], errors="coerce")/2_589_988.110336)
    df["pop_density_sqmi"] = (pd.to_numeric(df["pop_total"], errors="coerce")/area_sqmi).round(3)

/tmp/ipython-input-691763311.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["pop_density_sqmi"] = (pd.to_numeric(df["pop_total"], errors="coerce")/area_sqmi).round(3)


In [ ]:
id_cols   = [c for c in ["county_fips","state_name","state_abbr","county_name"] if c in df.columns]
rucc_cols = [c for c in ["rucc_code","rucc_metro_flag"] if c in df.columns]
ctrl_cols = [c for c in [
    "median_household_income_usd","per_capita_income_usd","unemployment_rate_pct",
    "pct_hs_or_higher","pct_bachelors_or_higher","pct_households_broadband",
    "pct_households_no_vehicle","pct_households_snap","mean_travel_time_to_work_min",
    "gini_index","pop_total","pop_density_sqmi"
] if c in df.columns]
n_counts  = [c for c in ["count_grocery_2021","count_supermarkets_2021","count_totalfoodstores_2021"] if c in df.columns]
n_rates   = [c for c in ["count_grocery_per10k_2021","count_supermarkets_per10k_2021","count_totalfoodstores_per10k_2021"] if c in df.columns]

eda_cols = [*id_cols, *rucc_cols, *ctrl_cols, *n_counts, *n_rates]
df_eda = df[[c for c in eda_cols if c in df.columns]].copy()

eda_subset_path = OUT / "acs_rucc_controls_nanda_2021_imputed_highfilter_eda_subset.csv"
df_eda.to_csv(eda_subset_path, index=False)
print("Saved EDA subset:", eda_subset_path, "| cols:", len(df_eda.columns))

Saved EDA subset: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_imputed_highfilter_eda_subset.csv | cols: 24


In [ ]:
per10k_cols = [c for c in df.columns if c.endswith("_per10k_2021") and c.startswith("count_")]
if per10k_cols:
    melt_cols = id_cols + rucc_cols
    tidy = df[melt_cols + per10k_cols].melt(id_vars=melt_cols, var_name="metric", value_name="per10k")
    tidy_path = OUT / "acs_rucc_controls_nanda_2021_imputed_highfilter_per10k_long.csv"
    tidy.to_csv(tidy_path, index=False)
    print("Saved per10k long:", tidy_path, "| rows:", len(tidy))
else:
    print("No per10k columns found; skipping tidy export.")

Saved per10k long: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_imputed_highfilter_per10k_long.csv | rows: 18864


In [ ]:
inv = pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str)})
inv_path = OUT / "acs_rucc_controls_nanda_2021_imputed_highfilter_column_inventory.csv"
inv.to_csv(inv_path, index=False)
print("Saved column inventory:", inv_path)

Saved column inventory: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_imputed_highfilter_column_inventory.csv


In [ ]:
eda_ready_path = OUT / "acs_rucc_controls_nanda_2021_imputed_highfilter_eda_ready.csv"
df.to_csv(eda_ready_path, index=False)
print("Saved EDA-ready master:", eda_ready_path)

Saved EDA-ready master: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_imputed_highfilter_eda_ready.csv


Will add more EDA ready transformations.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
FILE = OUT / "acs_rucc_controls_nanda_2021_imputed_highfilter.csv"
df = pd.read_csv(FILE, dtype=str)
df["county_fips"] = df["county_fips"].str.zfill(5)

In [ ]:
# families by name pattern
count21_cols  = [c for c in df.columns if c.startswith("count_") and c.endswith("_2021")]
emps21_cols   = [c for c in df.columns if c.startswith("emps_")  and c.endswith("_2021")]
rate10k_cols  = [c for c in df.columns if c.endswith("_per10k_2021")]
ratesqmi_cols = [c for c in df.columns if c.endswith("_per_sqmi_2021")]
pct_cols      = [c for c in df.columns if c.startswith("pct_") or c.endswith("_pct")]
usd_cols      = [c for c in df.columns if c.endswith("_usd")]
totals_cols   = [c for c in ["households_count","pop_total","totpop_2021","aland20_2021","land_area_sqmi","pop_density_sqmi"] if c in df.columns]
other_num     = [c for c in ["gini_index","mean_travel_time_to_work_min"] if c in df.columns]
miss_flags    = [c for c in df.columns if c.endswith("__is_missing")]

numeric_sets = [count21_cols, emps21_cols, rate10k_cols, ratesqmi_cols, pct_cols, usd_cols, totals_cols, other_num, miss_flags]

In [ ]:
def _clean_numeric_like(s: pd.Series) -> pd.Series:
    # drop commas, % signs, spaces; keep minus and decimal point
    return s.str.replace(r"[,\s%]", "", regex=True)

for cols in numeric_sets:
    for c in cols:
        df[c] = _clean_numeric_like(df[c].astype(str))

In [ ]:
# counts/employment/totals -> integer (nullable)
for c in set(count21_cols + emps21_cols + ["households_count","pop_total","totpop_2021"]):
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").round(0).astype("Int64")

# rates, $ medians, percentages, other continuous -> float
for c in set(rate10k_cols + ratesqmi_cols + usd_cols + pct_cols + ["gini_index","mean_travel_time_to_work_min","land_area_sqmi","pop_density_sqmi"]):
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# area in m² stays numeric float
if "aland20_2021" in df.columns:
    df["aland20_2021"] = pd.to_numeric(df["aland20_2021"], errors="coerce")

# missingness flags -> tiny ints (0/1)
for c in miss_flags:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("Int8")

In [ ]:
if "rucc_code" in df.columns:
    df["rucc_code"] = pd.to_numeric(df["rucc_code"], errors="coerce").astype("Int64")
for cat in ["rucc_metro_flag","state_abbr"]:
    if cat in df.columns:
        df[cat] = df[cat].astype("category")

In [ ]:
num_cols_all = [c for c in df.columns if df[c].dtype.kind in "if"]  # ints & floats
for c in num_cols_all:
    if df[c].dtype.kind == "f":
        df[c] = df[c].replace([np.inf, -np.inf], np.nan)

In [ ]:
def _is_numeric_dtype(s): return s.dtype.kind in "if"  # int or float
suspects = [c for c in set(sum(numeric_sets, [])) if c in df.columns and not _is_numeric_dtype(df[c])]
print("Non-numeric after coercion (should be 0):", len(suspects))
if suspects: print(suspects)

Non-numeric after coercion (should be 0): 0


In [ ]:
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M")

csv_out  = OUT / f"acs_rucc_controls_nanda_2021_imputed_highfilter_mathready_{stamp}.csv"
parq_out = OUT / f"acs_rucc_controls_nanda_2021_imputed_highfilter_mathready_{stamp}.parquet"

df.to_csv(csv_out, index=False)
print("Saved CSV:", csv_out)
try:
    df.to_parquet(parq_out, index=False)
    print("Saved Parquet:", parq_out)
except Exception as e:
    print("Parquet save skipped:", e)

Saved CSV: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_imputed_highfilter_mathready_20250927_0204.csv
Saved Parquet: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/acs_rucc_controls_nanda_2021_imputed_highfilter_mathready_20250927_0204.parquet


In [ ]:
import pandas as pd
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
OUT.mkdir(parents=True, exist_ok=True)

# <- change this if you want a different simple name
SIMPLE_NAME_CSV = "project_I_mathready.csv"
SIMPLE_NAME_PARQ = "project_I_mathready.parquet"

# save CSV
df.to_csv(OUT / SIMPLE_NAME_CSV, index=False)
print("Saved CSV:", OUT / SIMPLE_NAME_CSV)

# (optional) save Parquet for faster loads
try:
    df.to_parquet(OUT / SIMPLE_NAME_PARQ, index=False)
    print("Saved Parquet:", OUT / SIMPLE_NAME_PARQ)
except Exception as e:
    print("Parquet save skipped:", e)

# quick reload sanity (read-only)
test = pd.read_csv(OUT / SIMPLE_NAME_CSV, dtype=str)
print("Reloaded shape:", test.shape)


Saved CSV: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/project_I_mathready.csv
Saved Parquet: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/project_I_mathready.parquet
Reloaded shape: (3144, 159)


In [ ]:
print(list(df.columns))                 # all headers
print("Total cols:", len(df.columns))   # count
for i, c in enumerate(df.columns, 1):   # numbered list
    print(f"{i:3d}. {c}")

['county_fips', 'state_name', 'state_abbr', 'county_name', 'rucc_code', 'rucc_metro_flag', 'rucc_desc', 'pop_2020', 'households_count', 'pop_total', 'median_household_income_usd', 'per_capita_income_usd', 'unemployment_rate_pct', 'pct_hs_or_higher', 'pct_bachelors_or_higher', 'pct_households_broadband', 'pct_households_no_vehicle', 'pct_households_snap', 'mean_travel_time_to_work_min', 'gini_index', 'pct_one_race_white', 'pct_one_race_black_or_african_american', 'pct_one_race_american_indian_and_alaska_native', 'pct_one_race_asian', 'pct_one_race_native_hawaiian_and_other_pacific_islander', 'pct_one_race_some_other_race', 'pct_households_two_or_more_races', 'pct_households_hispanic_or_latino_origin_of_any_race', 'pct_households_white_alone_not_hispanic_or_latino', 'pct_families_families', 'pct_families_with_own_children_of_householder_under_18_years', 'pct_families_with_no_own_children_of_householder_under_18_years', 'pct_families_married_couple_families', 'pct_married_couple_families_

In [ ]:
# Cleansing file for EDA
import pandas as pd
from pathlib import Path

OUT = Path("/content/drive/MyDrive/GRADTDA5621_Project_I/Outputs")
df  = pd.read_csv(OUT / "project_I_mathready.csv", dtype=str)  # or your working file
df["county_fips"] = df["county_fips"].str.zfill(5)

# keep only a lean set that answers most EDA questions
id_cols = [c for c in ["county_fips","state_name","state_abbr","county_name"] if c in df.columns]
rucc    = [c for c in ["rucc_code","rucc_metro_flag"] if c in df.columns]
controls = [c for c in [
    "pop_total","households_count",
    "median_household_income_usd","per_capita_income_usd",
    "unemployment_rate_pct","pct_hs_or_higher","pct_bachelors_or_higher",
    "pct_households_broadband","pct_households_no_vehicle","pct_households_snap",
    "mean_travel_time_to_work_min","gini_index","pop_density_sqmi"
] if c in df.columns]

race_min = [c for c in [
    "pct_one_race_white","pct_one_race_black_or_african_american",
    "pct_households_hispanic_or_latino_origin_of_any_race"
] if c in df.columns]

# NaNDA counts (2021) and per-10k if present
nanda_counts = [c for c in ["count_grocery_2021","count_supermarkets_2021","count_totalfoodstores_2021"] if c in df.columns]
nanda_rates  = [c for c in ["count_grocery_per10k_2021","count_supermarkets_per10k_2021","count_totalfoodstores_per10k_2021"] if c in df.columns]

# exclude subgroup medians and __is_missing flags from the EDA view
exclude_flags = [c for c in df.columns if c.endswith("__is_missing")]
exclude_heavy_medians = [c for c in df.columns if c.startswith("median_") and c not in {
    "median_household_income_usd"  # keep overall median
}]

keep = [*id_cols, *rucc, *controls, *race_min, *nanda_counts, *nanda_rates]
eda_cols = [c for c in keep if c in df.columns]
eda_df = df[eda_cols].copy()

eda_path = OUT / "county_eda_min.csv"
eda_df.to_csv(eda_path, index=False)
print("Saved minimal EDA view:", eda_path, "| cols:", len(eda_df.columns))

Saved minimal EDA view: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/county_eda_min.csv | cols: 27


In [ ]:
short_map = {
    "median_household_income_usd":"med_hh_income",
    "per_capita_income_usd":"per_cap_income",
    "unemployment_rate_pct":"unemp_pct",
    "pct_hs_or_higher":"hs_plus_pct",
    "pct_bachelors_or_higher":"ba_plus_pct",
    "pct_households_broadband":"broadband_pct",
    "pct_households_no_vehicle":"no_vehicle_pct",
    "pct_households_snap":"snap_pct",
    "mean_travel_time_to_work_min":"commute_min",
    "gini_index":"gini",
    "pct_one_race_white":"white_pct",
    "pct_one_race_black_or_african_american":"black_pct",
    "pct_households_hispanic_or_latino_origin_of_any_race":"hispanic_pct",
    "count_grocery_2021":"grocery_21",
    "count_supermarkets_2021":"supermarkets_21",
    "count_totalfoodstores_2021":"totalstores_21",
    "count_grocery_per10k_2021":"grocery_per10k_21",
    "count_supermarkets_per10k_2021":"supermarkets_per10k_21",
    "count_totalfoodstores_per10k_2021":"totalstores_per10k_21",
}
eda_short = eda_df.rename(columns={k:v for k,v in short_map.items() if k in eda_df.columns})

eda_short_path = OUT / "county_eda_min_shortnames.csv"
eda_short.to_csv(eda_short_path, index=False)
print("Saved short-name EDA view:", eda_short_path)

Saved short-name EDA view: /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/county_eda_min_shortnames.csv


In [ ]:
per10k_cols = [c for c in eda_df.columns if c.endswith("_per10k_2021")]
if per10k_cols:
    melt_cols = [c for c in ["county_fips","state_abbr","county_name","rucc_metro_flag"] if c in eda_df.columns]
    tidy = eda_df[melt_cols + per10k_cols].melt(id_vars=melt_cols, var_name="metric", value_name="per10k")
    tidy_path = OUT / "county_eda_per10k_long.csv"
    tidy.to_csv(tidy_path, index=False)
    print("Saved per10k long (tidy):", tidy_path, "| rows:", len(tidy))
else:
    print("No per10k columns found; skipping tidy export.")

Saved per10k long (tidy): /content/drive/MyDrive/GRADTDA5621_Project_I/Outputs/county_eda_per10k_long.csv | rows: 9432
